# E23 association observer: public-four OFF → ON parity
Private diagnostic only. No training, GT scoring, submission.csv, or competition submission.
Same E23 model/source/settings. Public dummy videos are not a holdout.
Reuses the user's E23 notebook and pilkwang's CC0 support/model packs.


In [ ]:
import time
D3_STARTED = time.monotonic()


In [ ]:
import os
import numpy as np
from scipy.spatial import cKDTree

BIOHUB_PRESET = 'harmonic_mutual_support_association_fusion_v1'
BIOHUB_SCORE_AXIS = 'weighted harmonic forward/reverse association consensus on the fixed-90 dual-seed baseline'

# Parameters for constrained lineage reconstruction.
os.environ["BIOHUB_OUTPUT_FILTER_SHORT_TRACKS"] = "1"
os.environ["BIOHUB_DET_THRESHOLD"] = "0.96875"
os.environ["BIOHUB_MOTION_RELINK_LEARNED_BONUS"] = '1.0'
os.environ["BIOHUB_ILP_APPEARANCE_WEIGHT"] = "0.0"
os.environ["BIOHUB_ILP_DISAPPEARANCE_WEIGHT"] = "1.5"
os.environ["BIOHUB_ILP_DIVISION_WEIGHT"] = "1.0"  # REVIEW: reverted after testing -- 0.3/1.0/2.0/3.0 all scored 0.915 on the real leaderboard; local proxy dropped at 0.3 (messier graph, not more true divisions), so staying at the neutral, proven value
os.environ["BIOHUB_GAP_CLOSE_MAX_GAP"] = "2"
os.environ["BIOHUB_GAP_CLOSE_UM"] = "5.8"
os.environ["BIOHUB_GAP_DENSITY_ADAPTIVE"] = "1"
os.environ["BIOHUB_GAP_DENSITY_REFERENCE_UM"] = "6.5"
os.environ["BIOHUB_GAP_DENSITY_GAIN"] = "0.040"
os.environ["BIOHUB_GAP_DENSITY_MAX_STEP_DELTA_UM"] = "0.125"
os.environ["BIOHUB_GAP_DENSITY_NEIGHBORS"] = "3"
os.environ["BIOHUB_OUTPUT_MIN_TRACK_LEN"] = "6"
os.environ["BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS"] = "1"
os.environ["BIOHUB_OUTPUT_GAP2_RECOVERY"] = "0"
# REVIEW: reverted, was "4.66"  -- tested at 7.0, local proxy improved (first non-zero division_jaccard) but real score dropped to 0.914. Likely explained by validator/test-set division-content mismatch, not a validator bug -- back to the proven value.
os.environ["BIOHUB_SAFE_DIV_MAX_UM"] = "8.0"  # REVIEW: widened to match the 0.917 notebook directly. The earlier 7.0 test (pre-divergence-check) added noise that pure geometry couldn't filter out; divergence + mutual-NN are active now and should do that filtering instead.
os.environ["BIOHUB_SAFE_DIV_SISTER_MAX_UM"] = "11.0"  # REVIEW: matches 0.917, was "8.5"
os.environ["BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM"] = "10.0"  # REVIEW: matches 0.917, was "7.65"
os.environ["BIOHUB_SAFE_DIV_FRAME_FRAC_CAP"] = "0.0076"  # unchanged -- never bound at 0.920 (cap_skipped=0), leaving it as the safety net for this test
os.environ["BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP"] = "0.00375"  # unchanged, same reasoning
os.environ["BIOHUB_ADAPTIVE_SHORT_TRACK_RESCUE"] = "0"
os.environ["BIOHUB_USE_DEEPCENTER_VETO"] = "1"
os.environ["BIOHUB_REQUIRE_DEEPCENTER_VETO"] = "1"
os.environ["BIOHUB_DEEPCENTER_EXPECTED_EPOCH"] = "2"  # REVIEW: best.pt is epoch 2, not checkpoint_last.pt's epoch 500
os.environ["BIOHUB_DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM"] = "8.5"
os.environ["BIOHUB_DEEPCENTER_CHECKPOINT"] = "/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/weights/full_frame_center/best.pt"
os.environ["BIOHUB_DEEPCENTER_GAP_VETO"] = "1"
os.environ["BIOHUB_DEEPCENTER_GAP_THRESHOLD"] = "0.25"
os.environ["BIOHUB_DEEPCENTER_SAFE_DIV_VETO"] = '1'  # REVIEW: was 0; funnel telemetry showed safe-div accepting 81-100% of its own candidates -- turning on the one gate built specifically to catch that
os.environ["BIOHUB_RUN_OUTPUT_DIAGNOSTICS"] = "0"

# Exact candidate-arm settings formerly supplied by the diagnostic harness.
os.environ["BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT"] = "0.30"  # REVIEW: was 0.20 (0.915 reference value, never tuned). Harmonic fusion is the one mechanism with proven benefit (+0.002 in the reference notebook) -- testing whether more weight on the reverse-time evidence extracts more of it, rather than exploring another unproven lever.
os.environ["BIOHUB_BIDIRECTIONAL_FUSION_MODE"] = "harmonic_probability"
os.environ["BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION"] = "0.90"
os.environ["BIOHUB_DIAGNOSTIC_ARM"] = "harmonic_association_production"

print("BIOHUB_PRESET:", BIOHUB_PRESET)
print("BIOHUB_SCORE_AXIS:", BIOHUB_SCORE_AXIS)

In [ ]:
# Configuration guard: assert the single intended model-level change.
import json as _guard_json
import math as _guard_math
import os as _guard_os

_EXPECTED_NUMERIC = {
    "BIOHUB_DET_THRESHOLD": 0.96875,
    "BIOHUB_ILP_APPEARANCE_WEIGHT": 0.0,
    "BIOHUB_ILP_DISAPPEARANCE_WEIGHT": 1.5,
    "BIOHUB_GAP_CLOSE_UM": 5.8,
    "BIOHUB_OUTPUT_MIN_TRACK_LEN": 6.0,
    "BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT": 0.30,
}

_EXPECTED_TEXT = {
    "BIOHUB_BIDIRECTIONAL_FUSION_MODE": "harmonic_probability",
    "BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION": "0.90",
}

_drift = {}
for _key, _want in _EXPECTED_NUMERIC.items():
    _raw = _guard_os.environ.get(_key)
    if _raw is None:
        _drift[_key] = "missing"
        continue
    _got = float(_raw)
    if not _guard_math.isclose(_got, _want, rel_tol=0.0, abs_tol=1e-12):
        _drift[_key] = {"expected": _want, "actual": _got}

for _key, _want in _EXPECTED_TEXT.items():
    _got = _guard_os.environ.get(_key)
    if _got != _want:
        _drift[_key] = {"expected": _want, "actual": _got}

if _drift:
    raise RuntimeError(
        "Configuration drift detected: " + _guard_json.dumps(_drift, sort_keys=True)
    )

print("Configuration guard: PASS")
print("Baseline: fixed-90 dual-seed clean pipeline (public LB 0.913)")
print("Single model-level change: harmonic mutual-support association fusion")
print("Reverse-time association weight: 0.200")

In [ ]:
from __future__ import annotations

import csv
import importlib.util
import json
import math
import os
import shutil
import subprocess
import tempfile
import zipfile
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

COMPETITION = "biohub-cell-tracking-during-development"
COMP_DIR_CANDIDATES = [
    Path(f"/kaggle/input/competitions/{COMPETITION}"),
    Path(f"/kaggle/input/{COMPETITION}"),
]
COMP_DIR = next((path for path in COMP_DIR_CANDIDATES if path.exists()), COMP_DIR_CANDIDATES[0])
# Hard-bound production input: competition test images only.
TEST_DIR = COMP_DIR / "test"

WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
REPO_DIR = WORKING_DIR / "tracking_repo"
SUBMISSION_PATH = WORKING_DIR / "submission.csv"
RUN_STATS_PATH = WORKING_DIR / "run_stats.csv"

METHOD = "unet_transformer"
WEIGHTS_RELATIVE = f"weights/{METHOD}/split_0/edge_predictor_best.pth"
EXPERIMENT_TAG = "selected_101_dual_seed_near_balanced_center_confirmed_synthetic_gap"
TARGET_ARTIFACT_SLUG = os.environ.get("BIOHUB_TARGET_ARTIFACT_SLUG", "biohub-tracking-support-pack-50ep-v1")
PRIMARY_ARTIFACT_MANIFEST = Path(os.environ.get(
    "BIOHUB_PRIMARY_ARTIFACT_MANIFEST",
    "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/ARTIFACT_MANIFEST.json",
))
ALLOW_ARTIFACT_FALLBACK = os.environ.get("BIOHUB_ALLOW_ARTIFACT_FALLBACK", "0") != "0"

DET_THRESHOLD = float(os.environ.get("BIOHUB_DET_THRESHOLD", "0.99"))
UNET_BATCH_SIZE = int(os.environ.get("BIOHUB_UNET_BATCH_SIZE", "4"))
USE_ILP = os.environ.get("BIOHUB_USE_ILP", "1") != "0"
ILP_EDGE_WEIGHT = float(os.environ.get("BIOHUB_ILP_EDGE_WEIGHT", "-1.0"))
ILP_APPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_APPEARANCE_WEIGHT", "0.1"))
ILP_DISAPPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_DISAPPEARANCE_WEIGHT", "0.1"))
ILP_DIVISION_WEIGHT = float(os.environ.get("BIOHUB_ILP_DIVISION_WEIGHT", "1.0"))

# Hard-bound full production coverage. Internal GPU sharding remains exhaustive.
SLICE = ""

# If dependencies are not already installed and no offline wheels are attached,
# this controls whether the notebook attempts PyPI installation.
ALLOW_PIP_INSTALL = os.environ.get("BIOHUB_ALLOW_PIP_INSTALL", "0") != "0"
RUN_OUTPUT_DIAGNOSTICS = os.environ.get("BIOHUB_RUN_OUTPUT_DIAGNOSTICS", "1") != "0"

# Output-level graph post-processing.
OUTPUT_EDGE_MAX_UM = float(os.environ.get("BIOHUB_OUTPUT_EDGE_MAX_UM", "14.0"))
OUTPUT_ENFORCE_NEXT_FRAME = os.environ.get("BIOHUB_OUTPUT_ENFORCE_NEXT_FRAME", "1") != "0"
OUTPUT_SINGLE_PARENT_REPAIR = os.environ.get("BIOHUB_OUTPUT_SINGLE_PARENT_REPAIR", "1") != "0"
OUTPUT_SINGLE_CHILD_REPAIR = os.environ.get("BIOHUB_OUTPUT_SINGLE_CHILD_REPAIR", "0") != "0"
OUTPUT_PRUNE_ISOLATED = os.environ.get("BIOHUB_OUTPUT_PRUNE_ISOLATED", "1") != "0"
OUTPUT_MOTION_RELINK = os.environ.get("BIOHUB_OUTPUT_MOTION_RELINK", "1") != "0"
MOTION_RELINK_TIGHT_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_TIGHT_UM", "6.0"))
MOTION_RELINK_RELAXED_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_RELAXED_UM", "10.0"))
MOTION_RELINK_VELOCITY_WEIGHT = float(os.environ.get("BIOHUB_MOTION_RELINK_VELOCITY_WEIGHT", "0.5"))
MOTION_RELINK_LEARNED_BONUS = float(os.environ.get("BIOHUB_MOTION_RELINK_LEARNED_BONUS", "0.75"))
MOTION_RELINK_MAX_FRAME_NODES = int(os.environ.get("BIOHUB_MOTION_RELINK_MAX_FRAME_NODES", "2600"))

OUTPUT_DIVISION_GEOMETRY_FILTER = os.environ.get("BIOHUB_OUTPUT_DIVISION_GEOMETRY_FILTER", "0") != "0"
DIV_PARENT_MAX_UM = float(os.environ.get("BIOHUB_DIV_PARENT_MAX_UM", "10.5"))
DIV_SISTER_MAX_UM = float(os.environ.get("BIOHUB_DIV_SISTER_MAX_UM", "8.0"))
DIV_DROP_TO_SINGLE_IF_BAD = os.environ.get("BIOHUB_DIV_DROP_TO_SINGLE_IF_BAD", "1") != "0"
OUTPUT_GAP_CLOSE = os.environ.get("BIOHUB_OUTPUT_GAP_CLOSE", "1") != "0"
GAP_CLOSE_MAX_GAP = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_GAP", "1"))
GAP_CLOSE_UM = float(os.environ.get("BIOHUB_GAP_CLOSE_UM", "6.0"))
GAP_DENSITY_ADAPTIVE = os.environ.get("BIOHUB_GAP_DENSITY_ADAPTIVE", "0") != "0"
GAP_DENSITY_REFERENCE_UM = float(os.environ.get("BIOHUB_GAP_DENSITY_REFERENCE_UM", "6.5"))
GAP_DENSITY_GAIN = float(os.environ.get("BIOHUB_GAP_DENSITY_GAIN", "0.040"))
GAP_DENSITY_MAX_STEP_DELTA_UM = float(os.environ.get("BIOHUB_GAP_DENSITY_MAX_STEP_DELTA_UM", "0.125"))
GAP_DENSITY_NEIGHBORS = int(os.environ.get("BIOHUB_GAP_DENSITY_NEIGHBORS", "3"))
GAP_CLOSE_REUSE_EXISTING = os.environ.get("BIOHUB_GAP_CLOSE_REUSE_EXISTING", "1") != "0"
GAP_CLOSE_REUSE_UM = float(os.environ.get("BIOHUB_GAP_CLOSE_REUSE_UM", "3.2"))
GAP_CLOSE_MAX_ADDED_FRAC = float(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_FRAC", "0.05"))
GAP_CLOSE_MAX_ADDED_ABS = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_ABS", "2000"))
GAP_REFINE_SYNTHETIC = os.environ.get("BIOHUB_GAP_REFINE_SYNTHETIC", "1") != "0"
GAP_REFINE_WIN_Z = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_Z", "1"))
GAP_REFINE_WIN_YX = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_YX", "3"))
GAP_REFINE_MAX_SHIFT_UM = float(os.environ.get("BIOHUB_GAP_REFINE_MAX_SHIFT_UM", "3.2"))

OUTPUT_FILTER_SHORT_TRACKS = os.environ.get("BIOHUB_OUTPUT_FILTER_SHORT_TRACKS", "1") != "0"
OUTPUT_MIN_TRACK_LEN = int(os.environ.get("BIOHUB_OUTPUT_MIN_TRACK_LEN", "6"))
OUTPUT_KEEP_DIVISION_COMPONENTS = os.environ.get("BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS", "1") != "0"
ADAPTIVE_SHORT_TRACK_RESCUE = os.environ.get("BIOHUB_ADAPTIVE_SHORT_TRACK_RESCUE", "0") != "0"
SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC", "0.10"))
SHORT_TRACK_RESCUE_MIN_LEN = int(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MIN_LEN", "4"))
SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB", "0.82"))
SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM", "3.25"))
SHORT_TRACK_RESCUE_MAX_NODES_FRAC = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_FRAC", "0.018"))
SHORT_TRACK_RESCUE_MAX_NODES_ABS = int(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_ABS", "180"))

OUTPUT_LINEFIT_SMOOTH = os.environ.get("BIOHUB_OUTPUT_LINEFIT_SMOOTH", "1") != "0"
OUTPUT_LINEFIT_WEIGHT = float(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WEIGHT", "0.8"))
OUTPUT_LINEFIT_WINDOW = int(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WINDOW", "2"))

OUTPUT_GAP2_RECOVERY = os.environ.get("BIOHUB_OUTPUT_GAP2_RECOVERY", "0") != "0"
GAP2_MAX_TOTAL_UM = float(os.environ.get("BIOHUB_GAP2_MAX_TOTAL_UM", "10.2"))
GAP2_MAX_STEP_UM = float(os.environ.get("BIOHUB_GAP2_MAX_STEP_UM", "4.4"))
GAP2_MAX_LINKS_FRAC = float(os.environ.get("BIOHUB_GAP2_MAX_LINKS_FRAC", "0.0045"))
GAP2_MAX_LINKS_ABS = int(os.environ.get("BIOHUB_GAP2_MAX_LINKS_ABS", "180"))
GAP2_REQUIRE_CONTEXT = os.environ.get("BIOHUB_GAP2_REQUIRE_CONTEXT", "1") != "0"
GAP2_FRAME_FRAC_CAP = float(os.environ.get("BIOHUB_GAP2_FRAME_FRAC_CAP", "0.006"))

OUTPUT_SAFE_DIVISIONS = os.environ.get("BIOHUB_OUTPUT_SAFE_DIVISIONS", "1") != "0"
SAFE_DIV_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_MAX_UM", "8.0"))
SAFE_DIV_SISTER_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_SISTER_MAX_UM", "11.0"))
SAFE_DIV_EXISTING_CHILD_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM", "10.0"))
SAFE_DIV_FRAME_FRAC_CAP = float(os.environ.get("BIOHUB_SAFE_DIV_FRAME_FRAC_CAP", "0.008"))
SAFE_DIV_GLOBAL_FRAC_CAP = float(os.environ.get("BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP", "0.004"))

# --- new config: add next to the existing SAFE_DIV_* block ---
SAFE_DIV_DIVERGE_UM = float(os.environ.get("BIOHUB_SAFE_DIV_DIVERGE_UM", "2.25"))
SAFE_DIV_REQUIRE_DIVERGENCE = os.environ.get("BIOHUB_SAFE_DIV_REQUIRE_DIVERGENCE", "1") != "0"
SAFE_DIV_REQUIRE_MUTUAL_NN = os.environ.get("BIOHUB_SAFE_DIV_REQUIRE_MUTUAL_NN", "1") != "0"

# DeepCenter support is retained for compatibility, but this selected run keeps it disabled.
USE_DEEPCENTER_VETO = os.environ.get("BIOHUB_USE_DEEPCENTER_VETO", "1") != "0"
REQUIRE_DEEPCENTER_VETO = os.environ.get("BIOHUB_REQUIRE_DEEPCENTER_VETO", "1") != "0"
DEEPCENTER_MANIFEST_DEFAULT = os.environ.get(
    "BIOHUB_DEEPCENTER_MANIFEST_DEFAULT",
    "/kaggle/input/datasets/pilkwang/biohub-deepcenter-unet3d-center-prior-v1/ARTIFACT_MANIFEST.json",
)
DEEPCENTER_CHECKPOINT_DEFAULT = os.environ.get(
    "BIOHUB_DEEPCENTER_CHECKPOINT_DEFAULT",
    "/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/weights/full_frame_center/best.pt",
)
DEEPCENTER_RELATIVE = os.environ.get("BIOHUB_DEEPCENTER_RELATIVE", "weights/full_frame_center/best.pt")
DEEPCENTER_GAP_VETO = os.environ.get("BIOHUB_DEEPCENTER_GAP_VETO", "1") != "0"
DEEPCENTER_SAFE_DIV_VETO = os.environ.get("BIOHUB_DEEPCENTER_SAFE_DIV_VETO", "1") != "0"
DEEPCENTER_GAP_THRESHOLD = float(os.environ.get("BIOHUB_DEEPCENTER_GAP_THRESHOLD", "0.10"))
DEEPCENTER_EXPECTED_EPOCH = int(os.environ.get("BIOHUB_DEEPCENTER_EXPECTED_EPOCH", "0"))
DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM = float(os.environ.get("BIOHUB_DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM", "0"))
DEEPCENTER_SAFE_DIV_THRESHOLD = float(os.environ.get("BIOHUB_DEEPCENTER_SAFE_DIV_THRESHOLD", "0.12"))
DEEPCENTER_SCORE_WIN_Z = int(os.environ.get("BIOHUB_DEEPCENTER_SCORE_WIN_Z", "1"))
DEEPCENTER_SCORE_WIN_YX = int(os.environ.get("BIOHUB_DEEPCENTER_SCORE_WIN_YX", "2"))
DEEPCENTER_SCORE_CACHE_MAX_FRAMES = int(os.environ.get("BIOHUB_DEEPCENTER_SCORE_CACHE_MAX_FRAMES", "8"))

CONFIG_DISPLAY = {
    "experiment_tag": EXPERIMENT_TAG,
    "method": METHOD,
    "weights": WEIGHTS_RELATIVE,
    "target_artifact_slug": TARGET_ARTIFACT_SLUG,
    "primary_artifact_manifest": str(PRIMARY_ARTIFACT_MANIFEST),
    "allow_artifact_fallback": ALLOW_ARTIFACT_FALLBACK,
    "det_threshold": DET_THRESHOLD,
    "unet_batch_size": UNET_BATCH_SIZE,
    "use_ilp": USE_ILP,
    "ilp_edge_weight": ILP_EDGE_WEIGHT,
    "ilp_appearance_weight": ILP_APPEARANCE_WEIGHT,
    "ilp_disappearance_weight": ILP_DISAPPEARANCE_WEIGHT,
    "ilp_division_weight": ILP_DIVISION_WEIGHT,
    "slice": SLICE,
    "allow_pip_install": ALLOW_PIP_INSTALL,
    "output_edge_max_um": OUTPUT_EDGE_MAX_UM,
    "output_enforce_next_frame": OUTPUT_ENFORCE_NEXT_FRAME,
    "output_single_parent_repair": OUTPUT_SINGLE_PARENT_REPAIR,
    "output_single_child_repair": OUTPUT_SINGLE_CHILD_REPAIR,
    "output_prune_isolated": OUTPUT_PRUNE_ISOLATED,
    "output_motion_relink": OUTPUT_MOTION_RELINK,
    "motion_relink_tight_um": MOTION_RELINK_TIGHT_UM,
    "motion_relink_relaxed_um": MOTION_RELINK_RELAXED_UM,
    "motion_relink_velocity_weight": MOTION_RELINK_VELOCITY_WEIGHT,
    "motion_relink_learned_bonus": MOTION_RELINK_LEARNED_BONUS,
    "motion_relink_max_frame_nodes": MOTION_RELINK_MAX_FRAME_NODES,
    "output_division_geometry_filter": OUTPUT_DIVISION_GEOMETRY_FILTER,
    "div_parent_max_um": DIV_PARENT_MAX_UM,
    "div_sister_max_um": DIV_SISTER_MAX_UM,
    "div_drop_to_single_if_bad": DIV_DROP_TO_SINGLE_IF_BAD,
    "output_gap_close": OUTPUT_GAP_CLOSE,
    "gap_close_max_gap": GAP_CLOSE_MAX_GAP,
    "gap_close_effective_max_gap": min(GAP_CLOSE_MAX_GAP, 1),
    "gap_close_um": GAP_CLOSE_UM,
    "gap_density_adaptive": GAP_DENSITY_ADAPTIVE,
    "gap_density_reference_um": GAP_DENSITY_REFERENCE_UM,
    "gap_density_gain": GAP_DENSITY_GAIN,
    "gap_density_max_step_delta_um": GAP_DENSITY_MAX_STEP_DELTA_UM,
    "gap_density_neighbors": GAP_DENSITY_NEIGHBORS,
    "gap_close_reuse_existing": GAP_CLOSE_REUSE_EXISTING,
    "gap_close_reuse_um": GAP_CLOSE_REUSE_UM,
    "gap_close_max_added_frac": GAP_CLOSE_MAX_ADDED_FRAC,
    "gap_close_max_added_abs": GAP_CLOSE_MAX_ADDED_ABS,
    "gap_refine_synthetic": GAP_REFINE_SYNTHETIC,
    "gap_refine_win_z": GAP_REFINE_WIN_Z,
    "gap_refine_win_yx": GAP_REFINE_WIN_YX,
    "gap_refine_max_shift_um": GAP_REFINE_MAX_SHIFT_UM,
    "output_filter_short_tracks": OUTPUT_FILTER_SHORT_TRACKS,
    "output_min_track_len": OUTPUT_MIN_TRACK_LEN,
    "output_keep_division_components": OUTPUT_KEEP_DIVISION_COMPONENTS,
    "adaptive_short_track_rescue": ADAPTIVE_SHORT_TRACK_RESCUE,
    "short_track_rescue_trigger_removed_frac": SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC,
    "short_track_rescue_min_len": SHORT_TRACK_RESCUE_MIN_LEN,
    "short_track_rescue_min_mean_edge_prob": SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB,
    "short_track_rescue_max_mean_edge_dist_um": SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM,
    "short_track_rescue_max_nodes_frac": SHORT_TRACK_RESCUE_MAX_NODES_FRAC,
    "short_track_rescue_max_nodes_abs": SHORT_TRACK_RESCUE_MAX_NODES_ABS,
    "output_linefit_smooth": OUTPUT_LINEFIT_SMOOTH,
    "output_linefit_weight": OUTPUT_LINEFIT_WEIGHT,
    "output_linefit_window": OUTPUT_LINEFIT_WINDOW,
    "output_gap2_recovery": OUTPUT_GAP2_RECOVERY,
    "gap2_max_total_um": GAP2_MAX_TOTAL_UM,
    "gap2_max_step_um": GAP2_MAX_STEP_UM,
    "gap2_max_links_frac": GAP2_MAX_LINKS_FRAC,
    "gap2_max_links_abs": GAP2_MAX_LINKS_ABS,
    "gap2_require_context": GAP2_REQUIRE_CONTEXT,
    "gap2_frame_frac_cap": GAP2_FRAME_FRAC_CAP,
    "output_safe_divisions": OUTPUT_SAFE_DIVISIONS,
    "safe_div_max_um": SAFE_DIV_MAX_UM,
    "safe_div_sister_max_um": SAFE_DIV_SISTER_MAX_UM,
    "safe_div_existing_child_max_um": SAFE_DIV_EXISTING_CHILD_MAX_UM,
    "safe_div_frame_frac_cap": SAFE_DIV_FRAME_FRAC_CAP,
    "safe_div_global_frac_cap": SAFE_DIV_GLOBAL_FRAC_CAP,
    "use_deepcenter_add_only_gate": USE_DEEPCENTER_VETO,
    "deepcenter_gap_add_gate": DEEPCENTER_GAP_VETO,
    "deepcenter_safe_div_add_gate": DEEPCENTER_SAFE_DIV_VETO,
    "deepcenter_gap_threshold": DEEPCENTER_GAP_THRESHOLD,
    "deepcenter_expected_epoch": DEEPCENTER_EXPECTED_EPOCH,
    "deepcenter_gap_confirm_min_span_um": DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM,
    "deepcenter_safe_div_threshold": DEEPCENTER_SAFE_DIV_THRESHOLD,
    "deepcenter_checkpoint_default": DEEPCENTER_CHECKPOINT_DEFAULT,
}

print("Biohub learned UNet + node-transformer + ILP submission")
print("COMP_DIR:", COMP_DIR, "exists:", COMP_DIR.exists())
print("TEST_DIR:", TEST_DIR, "exists:", TEST_DIR.exists())
print(json.dumps(CONFIG_DISPLAY, indent=2, sort_keys=True))

In [ ]:
D3_EXPECTED_DATASETS = ['44b6_0113de3b', '44b6_0b24845f', '6bba_05b6850b', '6bba_05db0fb1']
assert sorted(p.stem for p in TEST_DIR.glob('*.zarr')) == D3_EXPECTED_DATASETS
assert not list(TEST_DIR.glob('*.geff')), 'GT must not be in prediction input'


In [ ]:
import re

os.environ.setdefault("POLARS_PREFER_PKG", "32")

PACKAGE_SPECS = {
    "tracksdata": ("tracksdata", "tracksdata"),
    "zarr": ("zarr", "zarr>=3.0.10,<4"),
    "pyscipopt": ("pyscipopt", "pyscipopt"),
    "geff": ("geff", "geff>=1.1.3.1.1"),
    "geff_spec": ("geff_spec", "geff-spec<1.2"),
    "ilpy": ("ilpy", "ilpy>=0.5.1"),
    "polars": ("polars", "polars>=1.36"),
    "blosc2": ("blosc2", "blosc2"),
    "dask": ("dask", "dask"),
    "imagecodecs": ("imagecodecs", "imagecodecs"),
    "skimage": ("skimage", "scikit-image>=0.24"),
    "pyarrow": ("pyarrow", "pyarrow"),
    "rustworkx": ("rustworkx", "rustworkx>=0.17.1"),
    "sqlalchemy": ("sqlalchemy", "sqlalchemy>=2"),
    "numcodecs": ("numcodecs", "numcodecs>=0.13,<0.16"),
    "donfig": ("donfig", "donfig>=0.8"),
    "google_crc32c": ("google_crc32c", "google-crc32c>=1.5"),
    "bidict": ("bidict", "bidict>=0.23.1"),
    "psygnal": ("psygnal", "psygnal>=0.14"),
    "rich": ("rich", "rich"),
    "networkx": ("networkx", "networkx>=3.2.1"),
    "pydantic": ("pydantic", "pydantic>=2.11"),
    "pydantic_core": ("pydantic_core", "pydantic-core"),
    "annotated_types": ("annotated_types", "annotated-types"),
    "typing_extensions": ("typing_extensions", "typing-extensions>=4.13"),
    "typing_inspection": ("typing_inspection", "typing-inspection"),
    "markdown_it": ("markdown_it", "markdown-it-py"),
    "pygments": ("pygments", "pygments"),
    "click": ("click", "click"),
    "cloudpickle": ("cloudpickle", "cloudpickle"),
    "fsspec": ("fsspec", "fsspec"),
    "partd": ("partd", "partd"),
    "locket": ("locket", "locket"),
    "toolz": ("toolz", "toolz"),
    "yaml": ("yaml", "pyyaml"),
    "ndindex": ("ndindex", "ndindex"),
    "msgpack": ("msgpack", "msgpack"),
    "numexpr": ("numexpr", "numexpr"),
    "deprecated": ("deprecated", "deprecated"),
    "wrapt": ("wrapt", "wrapt"),
    "imageio": ("imageio", "imageio"),
    "PIL": ("PIL", "pillow"),
    "tifffile": ("tifffile", "tifffile"),
    "lazy_loader": ("lazy_loader", "lazy-loader"),
    "tqdm": ("tqdm", "tqdm"),
}
EXTRA_SPECS_BY_NAME = {
    "tracksdata": ["bidict>=0.23.1", "psygnal>=0.14", "rich"],
    "zarr": ["donfig>=0.8", "google-crc32c>=1.5", "numcodecs>=0.13,<0.16"],
    "geff": ["geff-spec<1.2", "networkx>=3.2.1", "pydantic>=2.11", "numcodecs>=0.13,<0.16"],
    "geff_spec": ["pydantic>=2.11", "annotated-types", "pydantic-core", "typing-inspection"],
    "polars": ["polars-runtime-32"],
    "dask": ["click", "cloudpickle", "fsspec", "partd", "pyyaml", "toolz"],
    "partd": ["locket"],
    "blosc2": ["ndindex", "msgpack", "numexpr"],
    "numcodecs": ["deprecated", "msgpack", "wrapt"],
    "rich": ["markdown-it-py", "pygments"],
    "pydantic": ["annotated-types", "pydantic-core", "typing-extensions>=4.13", "typing-inspection"],
    "skimage": ["imageio", "pillow", "tifffile", "lazy-loader", "networkx"],
}
PIP_DEPENDENCIES = [spec for _, spec in PACKAGE_SPECS.values()]
REQUIRED_MODULES = {name: module for name, (module, _) in PACKAGE_SPECS.items() if module}
FALLBACK_ARTIFACT_SLUGS = ["biohub-tracking-support-pack-v1"]

# The safe path for offline reruns is to use attached wheels.
# Set BIOHUB_ALLOW_PIP_INSTALL=1 only for an interactive internet-enabled run.
ALLOW_PIP_INSTALL = os.environ.get("BIOHUB_ALLOW_PIP_INSTALL", "0") != "0"


def module_missing(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is None


def has_model_artifact(path: Path) -> bool:
    has_repo_dir = (path / "repo").exists()
    has_weights_dir = (path / "weights" / METHOD / "split_0" / "edge_predictor_best.pth").exists()
    has_repo_zip = (path / "repo.zip").exists()
    has_weights_zip = (path / "weights.zip").exists()
    return (has_repo_dir and has_weights_dir) or (has_repo_zip and has_weights_zip)


def artifact_manifest(path: Path) -> dict:
    manifest = path / "ARTIFACT_MANIFEST.json"
    if not manifest.exists():
        return {}
    try:
        return json.loads(manifest.read_text())
    except Exception:
        return {}


def artifact_matches_target(path: Path) -> bool:
    if ALLOW_ARTIFACT_FALLBACK:
        return True
    manifest = artifact_manifest(path)
    artifact_name = str(manifest.get("artifact_name", ""))
    path_text = str(path)
    return TARGET_ARTIFACT_SLUG in {artifact_name, path.name} or TARGET_ARTIFACT_SLUG in path_text


def candidate_roots_for_slug(slug: str) -> list[Path]:
    return [
        Path(f"/kaggle/input/datasets/pilkwang/{slug}"),
        Path(f"/kaggle/input/{slug}"),
        Path(f"/kaggle/input/{slug}/{slug}"),
        Path(f"PublicNotebook/{slug}"),
    ]


def find_artifacts_root() -> Path:
    candidates: list[Path] = []
    for env_name in ["BIOHUB_MODEL_ARTIFACTS", "BIOHUB_ARTIFACTS"]:
        explicit = os.environ.get(env_name, "").strip()
        if explicit:
            candidates.append(Path(explicit))

    candidates.append(PRIMARY_ARTIFACT_MANIFEST.parent)
    candidates.extend(candidate_roots_for_slug(TARGET_ARTIFACT_SLUG))

    if ALLOW_ARTIFACT_FALLBACK:
        for slug in FALLBACK_ARTIFACT_SLUGS:
            candidates.extend(candidate_roots_for_slug(slug))

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for child in input_root.iterdir():
            if not child.is_dir():
                continue
            child_text = str(child)
            if TARGET_ARTIFACT_SLUG in child_text or ALLOW_ARTIFACT_FALLBACK:
                candidates.append(child)
                candidates.append(child / child.name)
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.append(grandchild)

    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        if has_model_artifact(candidate) and artifact_matches_target(candidate):
            return candidate
    checked = "\n".join(str(path) for path in candidates[:80])
    raise FileNotFoundError(
        "Could not find the required model artifact. "
        f"Expected slug: {TARGET_ARTIFACT_SLUG}\n"
        "Attach the newly uploaded support dataset, or set BIOHUB_MODEL_ARTIFACTS.\n"
        "To debug with an older artifact, set BIOHUB_ALLOW_ARTIFACT_FALLBACK=1.\n"
        "Checked:\n" + checked
    )


def _has_package_file(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    patterns = ("*.whl", "*.tar.gz", "*.zip")
    return any(any(path.glob(pattern)) for pattern in patterns)


def find_offline_package_dirs(artifacts: Path) -> list[Path]:
    candidates: list[Path] = [
        artifacts / "wheels",
        artifacts,
        Path("/kaggle/working"),
        Path("/kaggle/working/wheels"),
    ]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for child in input_root.iterdir():
            if child.is_dir():
                candidates.extend([child / "wheels", child])
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.extend([grandchild / "wheels", grandchild])

    out: list[Path] = []
    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        if _has_package_file(candidate):
            out.append(candidate)
    return out


def purge_imported_modules(package_names: list[str]) -> None:
    roots = {"tracksdata"}
    for name in package_names:
        if name in PACKAGE_SPECS:
            module = PACKAGE_SPECS[name][0]
            roots.add(module.split(".")[0])
        if name == "polars":
            roots.add("polars")
    for root in roots:
        for module_name in list(sys.modules):
            if module_name == root or module_name.startswith(root + "."):
                sys.modules.pop(module_name, None)


def polars_runtime_ready() -> bool:
    try:
        import polars as _pl
        from polars._plr import PySeries as _PySeries

        _ = _PySeries
        return hasattr(_pl, "Float16") and _pl.Series([-999999.0], dtype=_pl.Float64).dtype == _pl.Float64
    except Exception:
        return False


def packages_requiring_refresh() -> list[str]:
    refresh: list[str] = []
    if not module_missing("polars") and not polars_runtime_ready():
        refresh.append("polars")

    if not module_missing("zarr"):
        try:
            import zarr as _zarr
            version_text = str(getattr(_zarr, "__version__", "0"))
            major = int(version_text.split(".", 1)[0])
            if major < 3:
                refresh.append("zarr")
        except Exception:
            refresh.append("zarr")
    return refresh


def dependency_specs_for(missing: list[str]) -> list[str]:
    specs: list[str] = []
    seen: set[str] = set()

    def add(spec: str) -> None:
        key = spec.lower()
        if key not in seen:
            seen.add(key)
            specs.append(spec)

    for name in missing:
        if name in PACKAGE_SPECS:
            add(PACKAGE_SPECS[name][1])
        for spec in EXTRA_SPECS_BY_NAME.get(name, []):
            add(spec)
    return specs


def import_failures() -> dict[str, str]:
    failures: dict[str, str] = {}
    for name, module_name in REQUIRED_MODULES.items():
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    return failures


def missing_names_from_failures(failures: dict[str, str]) -> list[str]:
    names: list[str] = []
    module_to_name = {module: name for name, module in REQUIRED_MODULES.items()}
    for message in failures.values():
        match = re.search(r"No module named ['\"]([^'\"]+)['\"]", message)
        if match:
            module = match.group(1).split(".")[0]
        else:
            match = re.search(r"module ['\"]([^'\"]+)['\"] has no attribute", message)
            if not match:
                continue
            module = match.group(1).split(".")[0]
        name = module_to_name.get(module)
        if name and name not in names:
            names.append(name)
    return names


def install_missing_dependencies(missing: list[str], artifacts: Path) -> None:
    specs = dependency_specs_for(missing)
    force_reinstall = bool({"polars", "zarr"} & set(missing))
    if not specs:
        return

    package_dirs = find_offline_package_dirs(artifacts)
    if package_dirs:
        offline_cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps"]
        if force_reinstall:
            offline_cmd.append("--force-reinstall")
        for package_dir in package_dirs:
            offline_cmd.extend(["--find-links", str(package_dir)])
        offline_cmd.extend(specs)
        print("Installing missing packages from offline package dirs:", missing)
        print("Dependency resolver is disabled with --no-deps to avoid replacing Kaggle numpy/scipy in a live kernel.")
        print("Offline package dirs:", [str(path) for path in package_dirs])
        result = subprocess.run(offline_cmd, text=True, capture_output=True)
        if result.returncode == 0:
            purge_imported_modules(missing)
            print("Offline dependency install succeeded.")
            return
        print("Offline dependency install failed. Last pip output:")
        print((result.stdout or "")[-2000:])
        print((result.stderr or "")[-2000:])

    if ALLOW_PIP_INSTALL:
        online_cmd = [sys.executable, "-m", "pip", "install", "--no-deps"]
        if force_reinstall:
            online_cmd.append("--force-reinstall")
        online_cmd.extend(specs)
        print("Installing missing packages from PyPI:", missing)
        result = subprocess.run(online_cmd, text=True, capture_output=True)
        if result.returncode == 0:
            purge_imported_modules(missing)
            print("PyPI dependency install succeeded.")
            return
        print("PyPI dependency install failed. Last pip output:")
        print((result.stdout or "")[-2000:])
        print((result.stderr or "")[-2000:])

    command = "pip install tracksdata zarr>=3.0.10,<4 pyscipopt geff geff-spec ilpy polars blosc2 dask imagecodecs pyarrow rustworkx sqlalchemy donfig numcodecs"
    raise ImportError(
        "Missing required packages or dependency wheels: " + ", ".join(missing) + "\n"
        "Attach the support dataset with offline wheels. If supplying Kaggle dependency input instead, use:\n"
        + command + "\n"
        "Do not quote zarr>=3.0.10,<4 in Kaggle dependency input."
    )


def ensure_dependencies(artifacts: Path) -> None:
    for _ in range(5):
        refresh = packages_requiring_refresh()
        if refresh:
            install_missing_dependencies(refresh, artifacts)
            continue

        missing = [pkg for pkg, module in REQUIRED_MODULES.items() if module_missing(module)]
        if missing:
            install_missing_dependencies(missing, artifacts)
            continue

        failures = import_failures()
        if not failures:
            print("Required graph/Zarr/ILP packages import successfully.")
            return

        missing_from_import = missing_names_from_failures(failures)
        if missing_from_import:
            install_missing_dependencies(missing_from_import, artifacts)
            continue

        raise ImportError(
            "Required packages are present but failed to import. "
            "This may indicate a binary dependency mismatch in the live notebook kernel. "
            "Keep Kaggle dependency input empty and attach the wheels artifact.\n"
            + json.dumps(failures, indent=2)
        )

    failures = import_failures()
    raise ImportError(
        "Dependency recovery did not converge after repeated offline installs. "
        "The attached support artifact may be missing wheels.\n"
        + json.dumps(failures, indent=2)
    )


def remove_path(path: Path) -> None:
    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.exists():
        shutil.rmtree(path)


def copy_or_extract_tree(src_dir: Path, src_zip: Path, dst: Path) -> None:
    remove_path(dst)
    if src_dir.exists() and src_dir.is_dir():
        shutil.copytree(src_dir, dst)
        return
    if src_zip.exists() and src_zip.is_file():
        dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(src_zip) as zf:
            zf.extractall(dst)
        return
    raise FileNotFoundError(f"Missing source tree or zip: {src_dir} / {src_zip}")


def link_or_copy_tree(src: Path, dst: Path) -> None:
    remove_path(dst)
    try:
        os.symlink(src, dst, target_is_directory=True)
    except Exception:
        shutil.copytree(src, dst)


def materialize_inference_repo(artifacts: Path) -> None:
    copy_or_extract_tree(artifacts / "repo", artifacts / "repo.zip", REPO_DIR)

    weights_src = artifacts / "weights"
    weights_zip = artifacts / "weights.zip"
    weights_dst = REPO_DIR / "weights"
    if weights_src.exists() and weights_src.is_dir():
        link_or_copy_tree(weights_src, weights_dst)
    elif weights_zip.exists() and weights_zip.is_file():
        remove_path(weights_dst)
        weights_dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(weights_zip) as zf:
            zf.extractall(weights_dst)
    else:
        raise FileNotFoundError(f"Missing weights tree or zip under {artifacts}")

    required = [
        REPO_DIR / "scripts" / "predict_unet_transformer.py",
        REPO_DIR / WEIGHTS_RELATIVE,
    ]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError("Materialized inference repo is incomplete:\n" + "\n".join(missing))
    print("Inference repo:", REPO_DIR)
    print("Weights:", REPO_DIR / WEIGHTS_RELATIVE)


ARTIFACTS = find_artifacts_root()
print("ARTIFACTS:", ARTIFACTS)
print("Has offline wheels:", (ARTIFACTS / "wheels").exists())
manifest_info = artifact_manifest(ARTIFACTS)
if manifest_info:
    print("Artifact name:", manifest_info.get("artifact_name"))
    print("Weight sha256:", manifest_info.get("model", {}).get("weight_sha256"))
    print("Weight path:", manifest_info.get("model", {}).get("weight_path"))
    _expected_primary_sha256 = "12f6881ee3620a831697ca098ff8f48e687a24225f4e048b538deec3562fe771"
    _actual_primary_sha256 = str(manifest_info.get("model", {}).get("weight_sha256", ""))
    if _actual_primary_sha256 != _expected_primary_sha256:
        raise RuntimeError(
            "Primary model checksum mismatch: "
            f"expected {_expected_primary_sha256}, got {_actual_primary_sha256 or 'missing'}"
        )

ensure_dependencies(ARTIFACTS)
materialize_inference_repo(ARTIFACTS)

# Verify every materialized support-repo Python source before the next cell
# performs the sealed dynamic inference patch. Also hash the actual primary and
# DeepCenter checkpoint bytes that the production path will use.
import hashlib as _integrity_hashlib

_support_expected_sha256 = {
    "scripts/augmentations.py": "13db09817bf492f8d0f710a0a4d09776320b262060167055090a303fc6057f4e",
    "scripts/dataspec.py": "e69bf952fb985477ac50ff8598a35020c95d20a035a09b81ab4056e655dd311f",
    "scripts/evaluate.py": "614813cc51c3581c6ccda4bb20725a19da8ecac4a27620654bfca58319cffa3c",
    "scripts/predict_unet_transformer.py": "c44e771ba5980b820f93091e03a303c25dfe8f3232e501f54dc9565731c234b9",
    "scripts/train_unet_transformer.py": "c4f6317736bb3bb1ec8f3f6e9a6d935a463e3f0f1f685481b2d13218d35dc9ea",
    "src/biohub_tracking/__init__.py": "26a18d8da84e40da73281a48ebc3017d847a2e57431ab63e8629d2109e6e8571",
    "src/biohub_tracking/division_metrics.py": "d1cf1e0a43009d02174f1699ce2aa28458a2220ac4b521731d3bcf31cf8c76be",
    "src/biohub_tracking/img_proc.py": "00e8ef0adc8b39f1aaaa547ea6197b906bf9e8c009e339d3e95f8f8dbf31be3f",
    "src/biohub_tracking/io.py": "efae135b088cecaab463d889f16c885ef6da3ad27b0747327d8ddc28d866b7bd",
    "src/biohub_tracking/metrics.py": "31baf45b54c78f68bab4f65dd8f4b38bca702abb644171c6df7c46cdeef55d83",
    "src/biohub_tracking/models/__init__.py": "ab7587ef79856bae50d24b62e5805092d0459ee1c586522b763f9ef70c093e1d",
    "src/biohub_tracking/models/simple_node_transformer.py": "b97209edeb03840e80d903e3e2a8c81c520641c8ef343f6ca2904d0f80db064e",
    "src/biohub_tracking/models/temporal_unet.py": "d809c35d42f504161074ddeaaa7aee5b407e5bca7f9b4e1d5f9b2ff345666cac"
}
_support_expected_manifest_sha256 = "978b626d1fd1e7397435a437dfe68691defe1572fc3c20e61012d7c9b52ed029"
_primary_expected_sha256 = "12f6881ee3620a831697ca098ff8f48e687a24225f4e048b538deec3562fe771"
_deepcenter_expected_sha256 = "8040999a92f6b7bbd98fa8cf458141e045c0f9ad7c936bdb3b18e1f7edafe2a0"  # best.pt (epoch 2), not checkpoint_last.pt (epoch 500)


def _integrity_sha256_file(path: Path) -> str:
    digest = _integrity_hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


_support_materialized_paths = {
    path.relative_to(REPO_DIR).as_posix(): path
    for path in REPO_DIR.rglob("*.py")
}
_support_actual_names = set(_support_materialized_paths)
_support_expected_names = set(_support_expected_sha256)
if _support_actual_names != _support_expected_names:
    raise RuntimeError({
        "support_repo_python_files_missing": sorted(
            _support_expected_names - _support_actual_names
        ),
        "support_repo_python_files_extra": sorted(
            _support_actual_names - _support_expected_names
        ),
    })
_support_actual_sha256 = {
    relative: _integrity_sha256_file(_support_materialized_paths[relative])
    for relative in sorted(_support_materialized_paths)
}
if _support_actual_sha256 != _support_expected_sha256:
    raise RuntimeError({
        "support_repo_python_checksum_mismatch": {
            relative: {
                "expected": _support_expected_sha256[relative],
                "actual": _support_actual_sha256[relative],
            }
            for relative in sorted(_support_expected_sha256)
            if _support_actual_sha256[relative]
            != _support_expected_sha256[relative]
        }
    })
_support_manifest_bytes = "".join(
    f"{_support_actual_sha256[relative]}  {relative}\n"
    for relative in sorted(_support_actual_sha256)
).encode("utf-8")
_support_actual_manifest_sha256 = _integrity_hashlib.sha256(
    _support_manifest_bytes
).hexdigest()
if _support_actual_manifest_sha256 != _support_expected_manifest_sha256:
    raise RuntimeError(
        "Support repo manifest checksum mismatch: "
        f"expected {_support_expected_manifest_sha256}, "
        f"got {_support_actual_manifest_sha256}"
    )

_primary_materialized_path = REPO_DIR / WEIGHTS_RELATIVE
_primary_actual_sha256 = _integrity_sha256_file(_primary_materialized_path)
if _primary_actual_sha256 != _primary_expected_sha256:
    raise RuntimeError(
        "Materialized primary model checksum mismatch: "
        f"expected {_primary_expected_sha256}, got {_primary_actual_sha256}"
    )

_deepcenter_candidate_strings = [
    os.environ.get("BIOHUB_DEEPCENTER_CHECKPOINT", "").strip(),
    "/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/weights/"
    "full_frame_center/best.pt",
    "/kaggle/input/datasets/pilkwang/biohub-deepcenter-unet3d-center-prior-v1/"
    "weights/full_frame_center/best.pt",
]
_deepcenter_candidates = []
for _candidate_string in _deepcenter_candidate_strings:
    if not _candidate_string:
        continue
    _candidate_path = Path(_candidate_string)
    if _candidate_path not in _deepcenter_candidates:
        _deepcenter_candidates.append(_candidate_path)
_deepcenter_materialized_path = next(
    (path for path in _deepcenter_candidates if path.is_file()),
    None,
)
if _deepcenter_materialized_path is None:
    raise FileNotFoundError({
        "missing_deepcenter_checkpoint": [str(path) for path in _deepcenter_candidates]
    })
_deepcenter_actual_sha256 = _integrity_sha256_file(
    _deepcenter_materialized_path
)
if _deepcenter_actual_sha256 != _deepcenter_expected_sha256:
    raise RuntimeError(
        "DeepCenter checkpoint checksum mismatch: "
        f"expected {_deepcenter_expected_sha256}, "
        f"got {_deepcenter_actual_sha256}"
    )
os.environ["BIOHUB_DEEPCENTER_CHECKPOINT"] = str(
    _deepcenter_materialized_path
)

print("Support repo Python manifest SHA256:", _support_actual_manifest_sha256)
print("Primary materialized SHA256:", _primary_actual_sha256)
print("DeepCenter materialized SHA256:", _deepcenter_actual_sha256)


# Resolve the independent-seed pack by checksum, then materialize only its weights.
import hashlib as _hashlib

_secondary_manifest_explicit = Path(os.environ.get(
    "BIOHUB_SECONDARY_ARTIFACT_MANIFEST",
    "/kaggle/input/datasets/pilkwang/biohub-temporal-unet3d-seed314159-v1/ARTIFACT_MANIFEST.json",
))
_secondary_expected_sha256 = "9bac2fa0dadc4a6fc1899e0caf187f4b553e0a7cd90ba1261a68b35ffe9e305f"
_secondary_slug = "biohub-temporal-unet3d-seed314159-v1"


def _find_secondary_artifact_root() -> tuple[Path, dict]:
    candidates = [
        _secondary_manifest_explicit,
        Path(f"/kaggle/input/{_secondary_slug}/ARTIFACT_MANIFEST.json"),
        Path(f"/kaggle/input/datasets/pilkwang/{_secondary_slug}/ARTIFACT_MANIFEST.json"),
    ]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        candidates.extend(input_root.rglob("ARTIFACT_MANIFEST.json"))

    seen = set()
    for manifest_path in candidates:
        manifest_path = manifest_path.expanduser()
        if manifest_path in seen or not manifest_path.is_file():
            continue
        seen.add(manifest_path)
        try:
            info = json.loads(manifest_path.read_text())
        except Exception:
            continue
        sha256 = str(info.get("model", {}).get("weight_sha256", ""))
        if sha256 == _secondary_expected_sha256:
            return manifest_path.parent, info
    raise FileNotFoundError(
        "Could not find the independent-seed artifact with weight SHA256 "
        + _secondary_expected_sha256
    )


SECONDARY_ARTIFACTS, secondary_manifest_info = _find_secondary_artifact_root()
SECONDARY_WEIGHTS_ROOT = WORKING_DIR / "secondary_seed_weights"
copy_or_extract_tree(
    SECONDARY_ARTIFACTS / "weights",
    SECONDARY_ARTIFACTS / "weights.zip",
    SECONDARY_WEIGHTS_ROOT,
)
SECONDARY_WEIGHTS_PATH = (
    SECONDARY_WEIGHTS_ROOT
    / "unet_transformer"
    / "split_0"
    / "edge_predictor_best.pth"
)
SECONDARY_CONFIG_PATH = SECONDARY_WEIGHTS_PATH.parent / "config.json"
for _required_secondary_path in (SECONDARY_WEIGHTS_PATH, SECONDARY_CONFIG_PATH):
    if not _required_secondary_path.is_file():
        raise FileNotFoundError(f"Missing secondary model file: {_required_secondary_path}")


def _sha256_file(path: Path) -> str:
    digest = _hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


_secondary_actual_sha256 = _sha256_file(SECONDARY_WEIGHTS_PATH)
if _secondary_actual_sha256 != _secondary_expected_sha256:
    raise RuntimeError(
        "Secondary model checksum mismatch: "
        f"expected {_secondary_expected_sha256}, got {_secondary_actual_sha256}"
    )

os.environ["BIOHUB_SECONDARY_WEIGHTS"] = str(SECONDARY_WEIGHTS_PATH)
os.environ["BIOHUB_SECONDARY_EDGE_WEIGHT"] = "0.15"
print("Secondary artifact:", SECONDARY_ARTIFACTS)
print("Secondary weight:", SECONDARY_WEIGHTS_PATH)
print("Secondary SHA256:", _secondary_actual_sha256)
print("Secondary edge-logit weight:", os.environ["BIOHUB_SECONDARY_EDGE_WEIGHT"])

os.environ["BIOHUB_SECONDARY_DETECTION_WEIGHT"] = "0.475"
os.environ["BIOHUB_SECONDARY_LINK_MODE"] = "low_margin_consensus"
os.environ["BIOHUB_SECONDARY_MIX_TEMPERATURE"] = "1"
os.environ["BIOHUB_SECONDARY_LOW_MARGIN_MAX"] = "0.35"
os.environ["BIOHUB_DUAL_SEED_EDGE_THRESHOLD"] = "0.48"

_runtime_integrity_receipt = {
    "status": "complete_label_free_runtime_integrity",
    "verified_before_dynamic_source_patch": True,
    "support_repo_python_file_count": len(_support_actual_sha256),
    "support_repo_python_sha256": _support_actual_sha256,
    "support_repo_python_manifest_sha256": _support_actual_manifest_sha256,
    "checkpoint_sha256": {
        "primary": _primary_actual_sha256,
        "secondary": _secondary_actual_sha256,
        "deepcenter": _deepcenter_actual_sha256,
    },
    "materialized_paths": {
        "primary": str(_primary_materialized_path),
        "secondary": str(SECONDARY_WEIGHTS_PATH),
        "deepcenter": str(_deepcenter_materialized_path),
    },
    "ground_truth_accessed": False,
}
_runtime_integrity_receipt_path = (
    WORKING_DIR / "bidirectional_production_runtime_integrity.json"
)
_runtime_integrity_receipt_path.write_text(
    json.dumps(_runtime_integrity_receipt, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print("Runtime integrity receipt:", _runtime_integrity_receipt_path)

In [ ]:
# Fail fast instead of silently running volumetric inference on CPU.
import torch as _torch

if not _torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is required for this notebook. Enable a Kaggle GPU accelerator and commit again."
    )
print("CUDA device:", _torch.cuda.get_device_name(0))

# Apply eight-view planar detection TTA before graph prediction.
_ps = REPO_DIR / "scripts" / "predict_unet_transformer.py"
_s = _ps.read_text()
_old = """        if cfg.det_tta:
            tta_flips = [(-1,), (-2,), (-2, -1)]
            for dims in tta_flips:
                imgs_flip = imgs.flip(dims)
                _, det_flip = model.encode(imgs_flip)
                for f in range(W):
                    det_logits[f] = det_logits[f] + det_flip[f].flip(dims)
                del imgs_flip, det_flip
            for f in range(W):
                det_logits[f] = det_logits[f] / 4"""
_new = """        if cfg.det_tta:
            _nv = 1
            for dims in [(-1,), (-2,), (-2, -1)]:
                imgs_flip = imgs.flip(dims)
                _, det_flip = model.encode(imgs_flip)
                for f in range(W):
                    det_logits[f] = det_logits[f] + det_flip[f].flip(dims)
                del imgs_flip, det_flip
                _nv += 1
            for _k in (1, 3):
                imgs_rot = torch.rot90(imgs, _k, dims=(-2, -1))
                _, det_rot = model.encode(imgs_rot)
                for f in range(W):
                    det_logits[f] = det_logits[f] + torch.rot90(det_rot[f], -_k, dims=(-2, -1))
                del imgs_rot, det_rot
                _nv += 1
            imgs_t = imgs.transpose(-1, -2)
            _, det_t = model.encode(imgs_t)
            for f in range(W):
                det_logits[f] = det_logits[f] + det_t[f].transpose(-1, -2)
            del imgs_t, det_t
            _nv += 1
            imgs_at = torch.rot90(imgs, 1, dims=(-2, -1)).transpose(-1, -2)
            _, det_at = model.encode(imgs_at)
            for f in range(W):
                det_logits[f] = det_logits[f] + torch.rot90(det_at[f].transpose(-1, -2), -1, dims=(-2, -1))
            del imgs_at, det_at
            _nv += 1
            for f in range(W):
                det_logits[f] = det_logits[f] / _nv"""
if _old in _s:
    _ps.write_text(_s.replace(_old, _new))
    print("TTA patch applied (400ep spatial D4-style)")
else:
    print("TTA WARNING: block not found - using default 4-way")

# Evaluate and calibrate two temporal models on one candidate graph.
_s = _ps.read_text()
_ensemble_replacements = [
    ('    downsample: tuple[int, ...] = (1, 4, 4),\n) -> tuple[np.ndarray, list[tuple[int, int, float, float]]]:', '    downsample: tuple[int, ...] = (1, 4, 4),\n    secondary_model: UNetNodeTransformer | None = None,\n    secondary_edge_weight: float = 0.0,\n    secondary_detection_weight: float = 0.0,\n    secondary_link_mode: str = "raw",\n    secondary_mix_temperature: float = 1.0,\n    secondary_low_margin_max: float = 0.2,\n) -> tuple[np.ndarray, list[tuple[int, int, float, float]]]:'),
    ('            for f in range(W):\n                det_logits[f] = det_logits[f] / _nv\n\n        del imgs', '            for f in range(W):\n                det_logits[f] = det_logits[f] / _nv\n\n        secondary_unet_out = None\n        if secondary_model is not None:\n            secondary_unet_out, secondary_det_logits = secondary_model.encode(imgs)\n\n            if secondary_detection_weight > 0.0:\n                if cfg.det_tta:\n                    _secondary_nv = 1\n                    for dims in [(-1,), (-2,), (-2, -1)]:\n                        secondary_imgs_flip = imgs.flip(dims)\n                        _, secondary_det_flip = secondary_model.encode(secondary_imgs_flip)\n                        for f in range(W):\n                            secondary_det_logits[f] = (\n                                secondary_det_logits[f] + secondary_det_flip[f].flip(dims)\n                            )\n                        del secondary_imgs_flip, secondary_det_flip\n                        _secondary_nv += 1\n                    for _k in (1, 3):\n                        secondary_imgs_rot = torch.rot90(imgs, _k, dims=(-2, -1))\n                        _, secondary_det_rot = secondary_model.encode(secondary_imgs_rot)\n                        for f in range(W):\n                            secondary_det_logits[f] = secondary_det_logits[f] + torch.rot90(\n                                secondary_det_rot[f], -_k, dims=(-2, -1)\n                            )\n                        del secondary_imgs_rot, secondary_det_rot\n                        _secondary_nv += 1\n                    secondary_imgs_t = imgs.transpose(-1, -2)\n                    _, secondary_det_t = secondary_model.encode(secondary_imgs_t)\n                    for f in range(W):\n                        secondary_det_logits[f] = (\n                            secondary_det_logits[f] + secondary_det_t[f].transpose(-1, -2)\n                        )\n                    del secondary_imgs_t, secondary_det_t\n                    _secondary_nv += 1\n                    secondary_imgs_at = torch.rot90(\n                        imgs, 1, dims=(-2, -1)\n                    ).transpose(-1, -2)\n                    _, secondary_det_at = secondary_model.encode(secondary_imgs_at)\n                    for f in range(W):\n                        secondary_det_logits[f] = secondary_det_logits[f] + torch.rot90(\n                            secondary_det_at[f].transpose(-1, -2),\n                            -1,\n                            dims=(-2, -1),\n                        )\n                    del secondary_imgs_at, secondary_det_at\n                    _secondary_nv += 1\n                    for f in range(W):\n                        secondary_det_logits[f] = secondary_det_logits[f] / _secondary_nv\n\n                for f in range(W):\n                    primary_det = det_logits[f]\n                    secondary_det = secondary_det_logits[f]\n                    primary_mean = primary_det.mean()\n                    secondary_mean = secondary_det.mean()\n                    primary_scale = primary_det.float().std(unbiased=False).clamp_min(1e-4)\n                    secondary_scale = secondary_det.float().std(unbiased=False).clamp_min(1e-4)\n                    scale_ratio = (primary_scale / secondary_scale).clamp(0.5, 2.0)\n                    secondary_det_aligned = (\n                        (secondary_det - secondary_mean) * scale_ratio + primary_mean\n                    )\n                    det_logits[f] = (\n                        (1.0 - secondary_detection_weight) * primary_det\n                        + secondary_detection_weight * secondary_det_aligned\n                    )\n\n            del secondary_det_logits\n\n        del imgs'),
    ('            edge_logits_pair = model.predict_edges(\n                unet_feat_src, unet_feat_tgt,\n                p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,\n                p_pos_src, p_pos_tgt,\n                p_mask_src, p_mask_tgt,\n            )  # (1, n_src, n_tgt)\n\n            raw = edge_logits_pair[0]', '            edge_logits_pair = model.predict_edges(\n                unet_feat_src, unet_feat_tgt,\n                p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,\n                p_pos_src, p_pos_tgt,\n                p_mask_src, p_mask_tgt,\n            )  # (1, n_src, n_tgt)\n\n            if secondary_model is not None:\n                if secondary_unet_out is None:\n                    raise RuntimeError("Secondary model is loaded but its feature map is missing")\n                secondary_feat_src = secondary_model._index_features(\n                    secondary_unet_out[:, f_idx], p_coords_src, p_mask_src,\n                )\n                secondary_feat_tgt = secondary_model._index_features(\n                    secondary_unet_out[:, f_idx + 1], p_coords_tgt, p_mask_tgt,\n                )\n                secondary_logits_pair = secondary_model.predict_edges(\n                    secondary_feat_src, secondary_feat_tgt,\n                    p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,\n                    p_pos_src, p_pos_tgt,\n                    p_mask_src, p_mask_tgt,\n                )\n\n                if secondary_link_mode == "raw":\n                    secondary_for_mix = secondary_logits_pair\n                    blend_weight = secondary_edge_weight\n                elif secondary_link_mode in {\n                    "calibrated", "adaptive", "low_margin_consensus"\n                }:\n                    primary_center = edge_logits_pair.mean(dim=1, keepdim=True)\n                    primary_scale = edge_logits_pair.float().std(\n                        dim=1, keepdim=True, unbiased=False\n                    ).clamp_min(1e-4)\n                    secondary_center = secondary_logits_pair.mean(dim=1, keepdim=True)\n                    secondary_scale = secondary_logits_pair.float().std(\n                        dim=1, keepdim=True, unbiased=False\n                    ).clamp_min(1e-4)\n                    secondary_scale_ratio = (primary_scale / secondary_scale).clamp(0.5, 2.0)\n                    secondary_for_mix = (\n                        (secondary_logits_pair - secondary_center) * secondary_scale_ratio\n                        + primary_center\n                    )\n                    if secondary_link_mode == "calibrated":\n                        blend_weight = secondary_edge_weight\n                    elif secondary_link_mode == "adaptive":\n                        if n_src >= 2:\n                            primary_probs = torch.softmax(edge_logits_pair[0], dim=0)\n                            secondary_probs = torch.softmax(secondary_for_mix[0], dim=0)\n                            primary_top2 = torch.topk(primary_probs, k=2, dim=0)\n                            secondary_top2 = torch.topk(secondary_probs, k=2, dim=0)\n                            primary_margin = primary_top2.values[0] - primary_top2.values[1]\n                            secondary_margin = secondary_top2.values[0] - secondary_top2.values[1]\n                            local_weight = (\n                                secondary_edge_weight + secondary_margin - primary_margin\n                            ).clamp(0.15, 0.75)\n                            same_parent = primary_top2.indices[0].eq(\n                                secondary_top2.indices[0]\n                            )\n                            local_weight = torch.where(\n                                same_parent,\n                                torch.maximum(\n                                    local_weight,\n                                    torch.full_like(local_weight, secondary_edge_weight),\n                                ),\n                                local_weight,\n                            )\n                            blend_weight = local_weight.view(1, 1, -1)\n                        else:\n                            blend_weight = secondary_edge_weight\n                    else:\n                        if n_src >= 2:\n                            primary_probs = torch.softmax(edge_logits_pair[0], dim=0)\n                            secondary_probs = torch.softmax(secondary_for_mix[0], dim=0)\n                            primary_top2 = torch.topk(primary_probs, k=2, dim=0)\n                            secondary_top2 = torch.topk(secondary_probs, k=2, dim=0)\n                            primary_margin = primary_top2.values[0] - primary_top2.values[1]\n                            same_parent = primary_top2.indices[0].eq(\n                                secondary_top2.indices[0]\n                            )\n                            uncertainty = (\n                                (secondary_low_margin_max - primary_margin)\n                                / secondary_low_margin_max\n                            ).clamp(0.0, 1.0)\n                            local_weight = secondary_edge_weight * uncertainty\n                            local_weight = torch.where(\n                                same_parent,\n                                local_weight,\n                                torch.zeros_like(local_weight),\n                            )\n                            blend_weight = local_weight.view(1, 1, -1)\n                        else:\n                            blend_weight = 0.0\n                else:\n                    raise ValueError(f"Unsupported secondary link mode: {secondary_link_mode}")\n\n                edge_logits_pair = (\n                    (1.0 - blend_weight) * edge_logits_pair\n                    + blend_weight * secondary_for_mix\n                )\n                if secondary_mix_temperature != 1.0:\n                    mixed_center = edge_logits_pair.mean(dim=1, keepdim=True)\n                    edge_logits_pair = mixed_center + (\n                        edge_logits_pair - mixed_center\n                    ) / secondary_mix_temperature\n\n            raw = edge_logits_pair[0]'),
    ('        del unet_out\n', '        del unet_out\n        if secondary_unet_out is not None:\n            del secondary_unet_out\n'),
    ('    model, window_size, downsample = load_model(weights_path, device)\n    print(', '    model, window_size, downsample = load_model(weights_path, device)\n\n    secondary_model = None\n    secondary_weights_text = os.environ.get("BIOHUB_SECONDARY_WEIGHTS", "").strip()\n    secondary_edge_weight = float(os.environ.get("BIOHUB_SECONDARY_EDGE_WEIGHT", "0"))\n    secondary_detection_weight = float(\n        os.environ.get("BIOHUB_SECONDARY_DETECTION_WEIGHT", "0")\n    )\n    secondary_link_mode = os.environ.get("BIOHUB_SECONDARY_LINK_MODE", "raw").strip()\n    secondary_mix_temperature = float(\n        os.environ.get("BIOHUB_SECONDARY_MIX_TEMPERATURE", "1")\n    )\n    secondary_low_margin_max = float(\n        os.environ.get("BIOHUB_SECONDARY_LOW_MARGIN_MAX", "0.2")\n    )\n    edge_candidate_threshold = float(\n        os.environ.get("BIOHUB_DUAL_SEED_EDGE_THRESHOLD", str(cfg.threshold))\n    )\n    if secondary_weights_text:\n        if not 0.0 < secondary_edge_weight < 1.0:\n            raise ValueError("BIOHUB_SECONDARY_EDGE_WEIGHT must be strictly between 0 and 1")\n        if not 0.0 <= secondary_detection_weight < 1.0:\n            raise ValueError(\n                "BIOHUB_SECONDARY_DETECTION_WEIGHT must be in the half-open interval [0, 1)"\n            )\n        if secondary_link_mode not in {\n            "raw", "calibrated", "adaptive", "low_margin_consensus"\n        }:\n            raise ValueError(\n                "BIOHUB_SECONDARY_LINK_MODE must be raw, calibrated, adaptive, "\n                "or low_margin_consensus"\n            )\n        if not 0.5 <= secondary_mix_temperature <= 2.0:\n            raise ValueError("BIOHUB_SECONDARY_MIX_TEMPERATURE must be in [0.5, 2.0]")\n        if not 0.0 < edge_candidate_threshold < 1.0:\n            raise ValueError("BIOHUB_DUAL_SEED_EDGE_THRESHOLD must be strictly between 0 and 1")\n        if not 0.0 < secondary_low_margin_max <= 1.0:\n            raise ValueError("BIOHUB_SECONDARY_LOW_MARGIN_MAX must be in (0, 1]")\n        secondary_model, secondary_window_size, secondary_downsample = load_model(\n            Path(secondary_weights_text), device,\n        )\n        if secondary_window_size != window_size or secondary_downsample != downsample:\n            raise ValueError(\n                "Primary and secondary models have incompatible inference grids: "\n                f"primary=(window={window_size}, downsample={downsample}), "\n                f"secondary=(window={secondary_window_size}, downsample={secondary_downsample})"\n            )\n        cfg.threshold = edge_candidate_threshold\n        print(\n            f"Secondary model: {secondary_weights_text} | "\n            f"edge weight={secondary_edge_weight:.3f} | "\n            f"detection weight={secondary_detection_weight:.3f} | "\n            f"link mode={secondary_link_mode} | "\n            f"temperature={secondary_mix_temperature:.3f} | "\n            f"low-margin max={secondary_low_margin_max:.3f} | "\n            f"edge threshold={cfg.threshold:.3f}",\n            flush=True,\n        )\n\n    print('),
    ('                unet_batch_size=unet_batch_size,\n                downsample=downsample,\n            )', '                unet_batch_size=unet_batch_size,\n                downsample=downsample,\n                secondary_model=secondary_model,\n                secondary_edge_weight=secondary_edge_weight,\n                secondary_detection_weight=secondary_detection_weight,\n                secondary_link_mode=secondary_link_mode,\n                secondary_mix_temperature=secondary_mix_temperature,\n                secondary_low_margin_max=secondary_low_margin_max,\n            )'),
]
for _patch_index, (_ensemble_old, _ensemble_new) in enumerate(
    _ensemble_replacements, start=1
):
    _ensemble_count = _s.count(_ensemble_old)
    if _ensemble_count != 1:
        raise RuntimeError(
            f'Calibrated dual-seed patch {_patch_index} expected one match, '
            f'found {_ensemble_count}'
        )
    _s = _s.replace(_ensemble_old, _ensemble_new, 1)
compile(_s, str(_ps), 'exec')
_ps.write_text(_s)
print('Calibrated dual-seed runtime patch applied')


# Frozen label-free frame retention guard.
os.environ["BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION"] = "0.90"
for _guard_old_log in WORKING_DIR.glob("retention_guard_*.jsonl"):
    _guard_old_log.unlink()

_s = _ps.read_text()
_guard_old = """                    det_logits[f] = (
                        (1.0 - secondary_detection_weight) * primary_det
                        + secondary_detection_weight * secondary_det_aligned
                    )"""
_guard_new = """                    blended_det = (
                        (1.0 - secondary_detection_weight) * primary_det
                        + secondary_detection_weight * secondary_det_aligned
                    )
                    primary_candidates = len(_detect_cells_pooled(
                        primary_det[0],
                        int(frame_indices[f]),
                        cfg.det_threshold,
                        pool_k,
                    ))
                    blended_candidates = len(_detect_cells_pooled(
                        blended_det[0],
                        int(frame_indices[f]),
                        cfg.det_threshold,
                        pool_k,
                    ))
                    minimum_retention = float(os.environ.get(
                        "BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION",
                        "0.90",
                    ))
                    candidate_retention = (
                        blended_candidates / primary_candidates
                        if primary_candidates
                        else 1.0
                    )
                    use_primary_detection = bool(
                        primary_candidates > 0
                        and candidate_retention < minimum_retention
                    )
                    det_logits[f] = (
                        primary_det if use_primary_detection else blended_det
                    )
                    if int(frame_indices[f]) not in seen_frames:
                        shard = os.environ.get(
                            "BIOHUB_GPU_SHARD", "single"
                        ).replace("/", "_")
                        guard_log = (
                            Path("/kaggle/working")
                            / f"retention_guard_{shard}.jsonl"
                        )
                        guard_record = {
                            "dataset": ds_path.stem,
                            "frame": int(frame_indices[f]),
                            "primary_candidates": int(primary_candidates),
                            "blended_candidates": int(blended_candidates),
                            "retention": float(candidate_retention),
                            "minimum_retention": float(minimum_retention),
                            "use_primary": bool(use_primary_detection),
                        }
                        with guard_log.open("a") as guard_handle:
                            guard_handle.write(
                                json.dumps(guard_record, sort_keys=True)
                                + "\\n"
                            )
                        if use_primary_detection:
                            print(
                                "BIOHUB_RETENTION_GUARD "
                                + json.dumps(guard_record, sort_keys=True),
                                flush=True,
                            )"""
_guard_matches = _s.count(_guard_old)
if _guard_matches != 1:
    raise RuntimeError(
        f"Retention guard expected one blend block, found {_guard_matches}"
    )
_s = _s.replace(_guard_old, _guard_new, 1)
compile(_s, str(_ps), "exec")
_ps.write_text(_s)
print(
    "Frozen frame retention guard applied at "
    + os.environ["BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION"]
)

# Fixed public reverse-time primary-association experiment. Detection, the
# secondary low-margin mix, ILP, and graph post-processing are unchanged.
import math as _bidirectional_math

_bidirectional_weight_guard = float(
    os.environ.get("BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT", "0")
)
if not _bidirectional_math.isclose(
    _bidirectional_weight_guard, 0.30, rel_tol=0.0, abs_tol=1e-12
):
    raise ValueError({
        "expected_bidirectional_weight": 0.30,
        "actual_bidirectional_weight": _bidirectional_weight_guard,
    })

_s = _ps.read_text()
_bi_old = '            edge_logits_pair = model.predict_edges(\n                unet_feat_src, unet_feat_tgt,\n                p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,\n                p_pos_src, p_pos_tgt,\n                p_mask_src, p_mask_tgt,\n            )  # (1, n_src, n_tgt)\n\n            if secondary_model is not None:\n'
_bi_new = '            edge_logits_pair = model.predict_edges(\n                unet_feat_src, unet_feat_tgt,\n                p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,\n                p_pos_src, p_pos_tgt,\n                p_mask_src, p_mask_tgt,\n            )  # (1, n_src, n_tgt)\n\n            _bidirectional_weight = float(\n                os.environ.get("BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT", "0")\n            )\n            if _bidirectional_weight > 0.0:\n                reverse_logits_native = model.predict_edges(\n                    unet_feat_tgt, unet_feat_src,\n                    p_coords_tgt * ds_arr_t, p_coords_src * ds_arr_t,\n                    p_pos_tgt, p_pos_src,\n                    p_mask_tgt, p_mask_src,\n                )  # (1, n_tgt, n_src)\n                reverse_logits_pair = reverse_logits_native.transpose(1, 2)\n\n                forward_center = edge_logits_pair.mean(dim=1, keepdim=True)\n                forward_scale = edge_logits_pair.float().std(\n                    dim=1, keepdim=True, unbiased=False\n                ).clamp_min(1e-4)\n                reverse_center = reverse_logits_pair.mean(dim=1, keepdim=True)\n                reverse_scale = reverse_logits_pair.float().std(\n                    dim=1, keepdim=True, unbiased=False\n                ).clamp_min(1e-4)\n                reverse_scale_ratio = (forward_scale / reverse_scale).clamp(0.5, 2.0)\n                reverse_scale_ratio = reverse_scale_ratio.to(reverse_logits_pair.dtype)\n                reverse_aligned = (\n                    (reverse_logits_pair - reverse_center) * reverse_scale_ratio\n                    + forward_center\n                )\n                # Biohub 145: require mutual forward/reverse support in probability space.\n                # The harmonic mean penalizes a candidate when either temporal direction\n                # assigns it very low probability, while calibration preserves the forward\n                # logit scale used by the unchanged downstream candidate threshold and ILP.\n                forward_prob = torch.softmax(edge_logits_pair.float(), dim=1).clamp_min(1e-8)\n                reverse_prob = torch.softmax(reverse_aligned.float(), dim=1).clamp_min(1e-8)\n                harmonic_prob = 1.0 / (\n                    (1.0 - _bidirectional_weight) / forward_prob\n                    + _bidirectional_weight / reverse_prob\n                )\n                harmonic_prob = harmonic_prob / harmonic_prob.sum(\n                    dim=1, keepdim=True\n                ).clamp_min(1e-8)\n                harmonic_logits = torch.log(harmonic_prob.clamp_min(1e-8))\n                harmonic_center = harmonic_logits.mean(dim=1, keepdim=True)\n                harmonic_scale = harmonic_logits.std(\n                    dim=1, keepdim=True, unbiased=False\n                ).clamp_min(1e-4)\n                harmonic_scale_ratio = (forward_scale / harmonic_scale).clamp(0.5, 2.0)\n                edge_logits_pair = (\n                    (harmonic_logits - harmonic_center) * harmonic_scale_ratio\n                    + forward_center\n                ).to(reverse_aligned.dtype)\n                del (\n                    reverse_logits_native,\n                    reverse_logits_pair,\n                    reverse_aligned,\n                    forward_prob,\n                    reverse_prob,\n                    harmonic_prob,\n                    harmonic_logits,\n                )\n            if secondary_model is not None:\n'
_bi_count = _s.count(_bi_old)
if _bi_count != 1:
    raise RuntimeError(
        f"Bidirectional edge patch expected one transformed block, found {_bi_count}"
    )
_s = _s.replace(_bi_old, _bi_new, 1)

_coordinate_manifest_old = '    coords = coords.astype(np.int16)\n    return coords, all_edges'
_coordinate_manifest_new = '    coords = coords.astype(np.int16)\n\n    # Label-free, pre-ILP detector-coordinate manifest. This executes inside\n    # predict_video, before build_graph and the ILP call in predict().\n    _coordinate_manifest_arm = os.environ.get(\n        "BIOHUB_DIAGNOSTIC_ARM", ""\n    ).strip()\n    if _coordinate_manifest_arm:\n        import hashlib as _coordinate_hashlib\n\n        _coordinate_shard = os.environ.get(\n            "BIOHUB_GPU_SHARD", "single"\n        ).replace("/", "_")\n        _coordinate_array = np.ascontiguousarray(\n            coords.astype("<i2", copy=False)\n        )\n        _coordinate_frame_counts = [\n            [int(_coordinate_t), int((_coordinate_array[:, 0] == _coordinate_t).sum())]\n            for _coordinate_t in np.unique(_coordinate_array[:, 0])\n        ]\n        _coordinate_record = {\n            "columns": ["t", "z", "y", "x"],\n            "coordinate_sha256": _coordinate_hashlib.sha256(\n                _coordinate_array.tobytes(order="C")\n            ).hexdigest(),\n            "dataset": ds_path.stem,\n            "dtype": "<i2",\n            "frame_counts": _coordinate_frame_counts,\n            "rows": int(len(_coordinate_array)),\n            "stage": "post_detection_pre_graph_pre_ilp",\n        }\n        _coordinate_manifest_path = (\n            Path("/kaggle/working")\n            / f"detector_coordinates_{_coordinate_manifest_arm}_"\n            f"{_coordinate_shard}.jsonl"\n        )\n        with _coordinate_manifest_path.open("a") as _coordinate_handle:\n            _coordinate_handle.write(\n                json.dumps(_coordinate_record, sort_keys=True) + "\\n"\n            )\n\n    return coords, all_edges'
_coordinate_manifest_count = _s.count(_coordinate_manifest_old)
if _coordinate_manifest_count != 1:
    raise RuntimeError(
        "Coordinate-manifest patch expected one pre-return block, found "
        f"{_coordinate_manifest_count}"
    )
_s = _s.replace(
    _coordinate_manifest_old, _coordinate_manifest_new, 1
)
compile(_s, str(_ps), "exec")
_ps.write_text(_s)
print(
    "Bidirectional harmonic-probability association fusion applied | weight=",
    _bidirectional_weight_guard,
)
print("Pre-ILP detector-coordinate manifest hook applied")



In [ ]:
import hashlib
D3_PAYLOAD = WORKING_DIR / "association_payload"
assert not D3_PAYLOAD.exists(), "fresh payload required"
D3_FILES = {'src/biohub/association_capture.py': '"""Lossless observation packets for the fixed E23 association path, not a predictor.\n\nThe caller supplies detached CPU NumPy arrays. No torch/model imports, network,\nlabels, score modification, implicit precision conversion, or top-k selection.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport re\nimport time\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\n\nMATRICES = (\n    "primary_forward_logits", "primary_reverse_logits", "secondary_forward_logits",\n    "mixed_logits", "mixed_probabilities",\n)\nFEATURES = ("primary_source_features", "primary_target_features",\n            "secondary_source_features", "secondary_target_features")\nCOORDINATES = ("source_indices", "target_indices", "source_coords_grid", "target_coords_grid")\n\n\n@dataclass(frozen=True)\nclass CaptureLimits:\n    nodes_per_side: int = 2048\n    pairs_per_packet: int = 1_000_000\n    pairs_per_run: int = 450_000_000\n    output_bytes: int = 12 * 1024**3\n    writer_seconds: float = 1200.0\n\n\ndef _require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef _sha(path):\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for block in iter(lambda: stream.read(1024**2), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef _json_bytes(value):\n    return (json.dumps(value, sort_keys=True, ensure_ascii=False, allow_nan=False) + "\\n").encode()\n\n\nclass PairCapture:\n    """Fresh-run writer with literal per-video detector counts and full-pair coverage.\n\n`primary_reverse_logits` must already be transposed into (source, target)\norientation by the observer. Original-space coordinates are grid * downsample;\nphysical coordinates additionally multiply scale. Neither conversion alters input.\n"""\n\n    def __init__(self, root: Path, frame_counts: dict[str, list[int]], *,\n                 downsample=(1, 4, 4), scale=(1.625, 0.40625, 0.40625), limits=None):\n        if limits is None:\n            limits = CaptureLimits()\n        _require(isinstance(limits, CaptureLimits), "invalid limits")\n        _require(all(type(getattr(limits, k)) is int and getattr(limits, k) > 0\n                     for k in ("nodes_per_side", "pairs_per_packet", "pairs_per_run", "output_bytes")),\n                 "limits must be positive integers")\n        _require(type(limits.writer_seconds) in (int, float) and np.isfinite(limits.writer_seconds)\n                 and limits.writer_seconds > 0, "invalid writer time limit")\n        _require(len(downsample) == 3 and all(type(v) is int and v > 0 for v in downsample),\n                 "invalid downsample")\n        _require(len(scale) == 3 and all(type(v) in (int, float) and np.isfinite(v) and v > 0 for v in scale),\n                 "invalid scale")\n        _require(bool(frame_counts), "empty dataset plan")\n        self.frame_counts = {}\n        self.offsets = {}\n        self.expected = set()\n        expected_pairs = 0\n        for name, counts in frame_counts.items():\n            _require(isinstance(name, str) and re.fullmatch(r"[A-Za-z0-9_-]+", name), "unsafe dataset name")\n            _require(len(counts) >= 2 and all(type(n) is int and 0 <= n <= limits.nodes_per_side for n in counts),\n                     "invalid detector count")\n            _require(len(counts) < 32768, "frame index exceeds int16")\n            self.frame_counts[name] = tuple(counts)\n            self.offsets[name] = np.cumsum([0, *counts], dtype=np.int64)\n            for t in range(len(counts) - 1):\n                size = counts[t] * counts[t + 1]\n                _require(size <= limits.pairs_per_packet, "packet pair budget exceeded")\n                expected_pairs += size\n                self.expected.add((name, t))\n        _require(expected_pairs <= limits.pairs_per_run, "run pair budget exceeded")\n        self.expected_pairs = expected_pairs\n        self.downsample, self.scale, self.limits = tuple(downsample), tuple(scale), limits\n        self.root = Path(root)\n        self.root.mkdir(parents=True, exist_ok=False)\n        self.records = {}\n        self.output_bytes = 0\n        self.writer_seconds = 0.0\n        self.failed = False\n        self.closed = False\n\n    def _available(self):\n        _require(not self.failed and not self.closed, "capture already failed or closed")\n\n    def _failure(self, exc):\n        self.failed = True\n        path = self.root / "ERROR.json"\n        if not path.exists():\n            with path.open("xb") as stream:\n                stream.write(_json_bytes({"status": "ERROR", "type": type(exc).__name__, "error": str(exc),\n                                          "submission_authorized": False}))\n\n    def _validate(self, dataset, t_source, arrays):\n        _require(type(t_source) is int and (dataset, t_source) in self.expected, "unexpected frame pair")\n        _require((dataset, t_source) not in self.records, "duplicate frame pair")\n        ns, nt = self.frame_counts[dataset][t_source:t_source + 2]\n        empty = ns == 0 or nt == 0\n        keys = set(COORDINATES) if empty else set(COORDINATES + MATRICES + FEATURES)\n        _require(set(arrays) == keys, "packet key set mismatch")\n        for key, a in arrays.items():\n            _require(isinstance(a, np.ndarray) and not a.dtype.hasobject, f"not a numeric ndarray: {key}")\n            if key in MATRICES:\n                shape, dtype = (ns, nt), np.dtype("float32")\n            else:\n                n = ns if "source" in key else nt\n                shape = (n,) if key.endswith("indices") else (n, 4) if key.endswith("grid") else (n, 32)\n                dtype = np.dtype("int64") if key.endswith("indices") else (\n                    np.dtype("int16") if key.endswith("grid") else np.dtype("float32"))\n            _require(a.shape == shape and a.dtype == dtype, f"shape/dtype mismatch: {key}")\n            _require(np.isfinite(a).all(), f"nonfinite: {key}")\n        for side, t in (("source", t_source), ("target", t_source + 1)):\n            start, end = self.offsets[dataset][t:t + 2]\n            _require(np.array_equal(arrays[f"{side}_indices"], np.arange(start, end, dtype=np.int64)),\n                     f"global detector ID mapping mismatch: {side}")\n            coords = arrays[f"{side}_coords_grid"]\n            _require(np.all(coords[:, 0] == t) and np.all(coords[:, 1:] >= 0), f"invalid t/grid coords: {side}")\n        if not empty:\n            probs = arrays["mixed_probabilities"]\n            _require(np.all((probs >= 0) & (probs <= 1)), "probability out of range")\n            _require(np.allclose(probs.sum(axis=0, dtype=np.float64), 1., atol=2e-5, rtol=0),\n                     "probabilities not normalised over source axis")\n        return ns, nt, empty\n\n    def write_pair(self, dataset: str, t_source: int, arrays: dict[str, np.ndarray]):\n        self._available()\n        started = time.perf_counter()\n        try:\n            ns, nt, empty = self._validate(dataset, t_source, arrays)\n            metadata = {"schema_version": "biohub.e23.association_pair.v1", "dataset": dataset,\n                        "t_source": t_source, "t_target": t_source + 1, "window_size": 2,\n                        "downsample_zyx": self.downsample, "scale_zyx_um": self.scale,\n                        "matrix_axes": ["source", "target"], "probability_normalisation_axis": "source",\n                        "status": "SKIPPED_EMPTY" if empty else "OBSERVED",\n                        "n_source": ns, "n_target": nt, "dense_pairs": ns * nt,\n                        "arrays": {k: {"shape": list(a.shape), "dtype": a.dtype.str,\n                                       "sha256": hashlib.sha256(a.tobytes(order="C")).hexdigest()}\n                                   for k, a in arrays.items()}, "submission_authorized": False}\n            folder = self.root / dataset\n            folder.mkdir(exist_ok=True)\n            path = folder / f"pair_{t_source:04d}.npz"\n            with path.open("xb") as stream:\n                np.savez_compressed(stream, metadata=np.array(_json_bytes(metadata).decode()), **arrays)\n            with np.load(path, allow_pickle=False) as saved:\n                _require(set(saved.files) == set(arrays) | {"metadata"}, "roundtrip key mismatch")\n                _require(json.loads(str(saved["metadata"])) == json.loads(json.dumps(metadata)),\n                         "roundtrip metadata mismatch")\n                for key, original in arrays.items():\n                    actual = saved[key]\n                    _require(actual.dtype == original.dtype and actual.shape == original.shape and\n                             actual.tobytes(order="C") == original.tobytes(order="C"),\n                             f"lossy roundtrip: {key}")\n            digest = _sha(path)\n            self.output_bytes += path.stat().st_size\n            self.writer_seconds += time.perf_counter() - started\n            _require(self.output_bytes <= self.limits.output_bytes, "output byte budget exceeded")\n            _require(self.writer_seconds <= self.limits.writer_seconds, "writer time budget exceeded")\n            self.records[dataset, t_source] = {"path": path.relative_to(self.root).as_posix(),\n                                             "bytes": path.stat().st_size, "sha256": digest, **metadata}\n            return json.loads(json.dumps(self.records[dataset, t_source]))\n        except Exception as exc:\n            self._failure(exc)\n            raise\n\n    def finish(self):\n        self._available()\n        started = time.perf_counter()\n        try:\n            _require(set(self.records) == self.expected, "incomplete frame-pair capture")\n            _require(sum(r["dense_pairs"] for r in self.records.values()) == self.expected_pairs,\n                     "dense pair count mismatch")\n            for record in self.records.values():\n                path = self.root / record["path"]\n                _require(path.stat().st_size == record["bytes"] and _sha(path) == record["sha256"],\n                         "saved packet changed")\n            self.writer_seconds += time.perf_counter() - started\n            _require(self.writer_seconds <= self.limits.writer_seconds, "writer time budget exceeded")\n            result = {"schema_version": "biohub.e23.association_capture.v1", "status": "PAIR_CAPTURE_COMPLETE",\n                      "capture_only": True, "submission_authorized": False,\n                      "graph_ID_mapping_complete": False, "inference_parity_verified": False,\n                      "frame_counts": self.frame_counts, "dense_pairs": self.expected_pairs,\n                      "writer_seconds": self.writer_seconds, "packet_bytes": self.output_bytes,\n                      "records": [self.records[key] for key in sorted(self.records)]}\n            encoded = _json_bytes(result)\n            _require(self.output_bytes + len(encoded) <= self.limits.output_bytes, "manifest byte budget exceeded")\n            with (self.root / "MANIFEST.json").open("xb") as stream:\n                stream.write(encoded)\n            self.closed = True\n            return result\n        except Exception as exc:\n            self._failure(exc)\n            raise\n', 'src/biohub/association_observer.py': '"""Sequential E23 observation adapter; no predictor, GT, model load or network."""\n\nfrom __future__ import annotations\n\nimport json\nimport time\nfrom functools import wraps\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom biohub.association_capture import PairCapture, _json_bytes, _require, _sha\n\n\ndef observed(method):\n    """Account synchronous capture work at call boundaries; never swallow errors."""\n    @wraps(method)\n    def call(self, *args, **kwargs):\n        _require(not self.failed and not self.closed, "observer failed or closed")\n        started = time.perf_counter()\n        try:\n            result = method(self, *args, **kwargs)\n            self.seconds += time.perf_counter() - started\n            _require(self.seconds <= self.pairs.limits.writer_seconds, "observer time budget exceeded")\n            _require(self.pairs.output_bytes + self.graph_bytes <= self.pairs.limits.output_bytes,\n                     "observer byte budget exceeded")\n            return result\n        except Exception as exc:\n            self.failed = True\n            path = self.root / "ERROR.json"\n            if not path.exists():\n                with path.open("xb") as stream:\n                    stream.write(_json_bytes({"status": "ERROR", "error": str(exc),\n                                              "submission_authorized": False}))\n            raise\n    return call\n\n\ndef _copy_tensor(value, *, batched=True):\n    # No .float(), autocast, RNG, or model invocation. PairCapture checks actual dtype.\n    array = value.detach().cpu().numpy()\n    if batched:\n        _require(array.ndim == 3 and array.shape[0] == 1, "expected batch size one")\n        array = array[0]\n    return array.copy()\n\n\nclass AssociationObserver:\n    def __init__(self, root: Path, frame_counts, *, limits=None, downsample=(1, 4, 4),\n                 scale=(1.625, .40625, .40625)):\n        self.root = Path(root)\n        # PairCapture validates the plan before creating any directory.\n        _require(not self.root.exists(), "observer output already exists")\n        self.pairs = PairCapture(self.root / "pairs", frame_counts, limits=limits,\n                                 downsample=downsample, scale=scale)\n        self.failed = self.closed = False\n        self.seconds = self.graph_bytes = 0\n        self.dataset = None\n        self.pending = None\n        self.stage = "idle"\n        self.completed = set()\n        self.artifacts = []\n        self.coords = self.edges = self.before = None\n\n    @observed\n    def start_video(self, name, frames, window_size, downsample, scale, activation, *,\n                    threshold=.48, max_parents=None, max_children=None):\n        _require(self.dataset is None and name not in self.completed, "video overlap or duplicate")\n        _require(name in self.pairs.frame_counts and frames == len(self.pairs.frame_counts[name]),\n                 "video plan mismatch")\n        _require(window_size == 2 and tuple(downsample) == self.pairs.downsample\n                 and tuple(scale) == self.pairs.scale and activation == "softmax", "inference geometry mismatch")\n        _require(threshold == .48 and max_parents is None and max_children is None,\n                 "E23 candidate filtering mismatch")\n        self.threshold = threshold\n        self.dataset, self.stage = name, "pair"\n\n    @observed\n    def start_pair(self, t, s_src, e_src, s_tgt, e_tgt, coords):\n        _require(self.stage == "pair" and self.pending is None, "pair stage order mismatch")\n        _require((self.dataset, t) in self.pairs.expected, "unexpected pair")\n        offsets = self.pairs.offsets[self.dataset]\n        _require((s_src, e_src, s_tgt, e_tgt) == (offsets[t], offsets[t + 1], offsets[t + 1], offsets[t + 2]),\n                 "detector offsets mismatch")\n        self.t = t\n        self.pending = {\n            "source_indices": np.arange(s_src, e_src, dtype=np.int64),\n            "target_indices": np.arange(s_tgt, e_tgt, dtype=np.int64),\n            "source_coords_grid": coords[s_src:e_src].copy(),\n            "target_coords_grid": coords[s_tgt:e_tgt].copy(),\n        }\n        if s_src == e_src or s_tgt == e_tgt:\n            self.pairs.write_pair(self.dataset, t, self.pending)\n            self.pending = None\n        else:\n            self.stage = "primary"\n\n    @observed\n    def primary(self, logits, source, target):\n        _require(self.stage == "primary", "primary stage order mismatch")\n        self.pending.update(primary_forward_logits=_copy_tensor(logits),\n                            primary_source_features=_copy_tensor(source),\n                            primary_target_features=_copy_tensor(target))\n        self.stage = "reverse"\n\n    @observed\n    def reverse(self, transposed_logits):\n        _require(self.stage == "reverse", "reverse stage order mismatch")\n        self.pending["primary_reverse_logits"] = _copy_tensor(transposed_logits)\n        self.stage = "secondary"\n\n    @observed\n    def secondary(self, logits, source, target):\n        _require(self.stage == "secondary", "secondary stage order mismatch")\n        self.pending.update(secondary_forward_logits=_copy_tensor(logits),\n                            secondary_source_features=_copy_tensor(source),\n                            secondary_target_features=_copy_tensor(target))\n        self.stage = "mixed"\n\n    @observed\n    def mixed(self, raw, probabilities):\n        _require(self.stage == "mixed", "mixed stage order mismatch")\n        self.pending.update(mixed_logits=_copy_tensor(raw, batched=False),\n                            mixed_probabilities=probabilities.copy())\n        self.pairs.write_pair(self.dataset, self.t, self.pending)\n        self.pending, self.stage = None, "pair"\n\n    @observed\n    def end_video(self, coords, edges):\n        _require(self.stage == "pair" and self.pending is None, "incomplete pair")\n        name = self.dataset\n        _require({(name, t) for t in range(len(self.pairs.frame_counts[name]) - 1)}\n                 <= set(self.pairs.records), "incomplete video")\n        _require(coords.dtype == np.int16 and coords.shape == (sum(self.pairs.frame_counts[name]), 4),\n                 "returned coordinates shape/dtype mismatch")\n        by_time = {t: {} for t in range(len(self.pairs.frame_counts[name]) - 1)}\n        for s, target, probability, distance in edges:\n            _require(type(s) is int and type(target) is int and 0 <= s < len(coords)\n                     and 0 <= target < len(coords), "invalid returned edge index")\n            t = int(coords[s, 0])\n            _require(t in by_time and coords[target, 0] == t + 1, "invalid returned edge time")\n            _require((s, target) not in by_time[t], "duplicate returned edge")\n            by_time[t][s, target] = (probability, distance)\n        for t in range(len(self.pairs.frame_counts[name]) - 1):\n            record = self.pairs.records[name, t]\n            with np.load(self.pairs.root / record["path"], allow_pickle=False) as packet:\n                for side in ("source", "target"):\n                    grid = packet[f"{side}_coords_grid"].astype(np.int64)\n                    grid[:, 1:] *= self.pairs.downsample\n                    _require(np.array_equal(coords[packet[f"{side}_indices"]], grid),\n                             "returned coordinates differ from observed detections")\n                expected = {}\n                if record["status"] != "SKIPPED_EMPTY":\n                    probabilities = packet["mixed_probabilities"]\n                    source_ids, target_ids = packet["source_indices"], packet["target_indices"]\n                    source_grid, target_grid = packet["source_coords_grid"], packet["target_coords_grid"]\n                    for i, j in zip(*np.where(probabilities > self.threshold), strict=True):\n                        delta = source_grid[i, 1:].astype(np.float32) - target_grid[j, 1:].astype(np.float32)\n                        expected[int(source_ids[i]), int(target_ids[j])] = (\n                            float(probabilities[i, j]), float(np.linalg.norm(delta)))\n                _require(expected == by_time[t], "returned candidates differ from observed probability/geometry")\n        self.coords = coords.copy()\n        self.edges = tuple(tuple(edge) for edge in edges)\n        self.stage = "pre_graph"\n\n    def _save_arrays(self, label, arrays):\n        _require(all(isinstance(a, np.ndarray) and not a.dtype.hasobject for a in arrays.values()),\n                 "graph array type mismatch")\n        path = self.root / f"{self.dataset}_{label}.npz"\n        with path.open("xb") as stream:\n            np.savez_compressed(stream, **arrays)\n        with np.load(path, allow_pickle=False) as saved:\n            _require(set(saved.files) == set(arrays), "graph roundtrip keys mismatch")\n            for key, original in arrays.items():\n                actual = saved[key]\n                _require(actual.dtype == original.dtype and actual.shape == original.shape\n                         and actual.tobytes() == original.tobytes(), "graph roundtrip mismatch")\n        self.graph_bytes += path.stat().st_size\n        self.artifacts.append({"path": path.name, "bytes": path.stat().st_size, "sha256": _sha(path),\n                               "dataset": self.dataset, "stage": label})\n\n    @staticmethod\n    def _graph_tables(graph):\n        nodes = graph.node_attrs(attr_keys=["node_id", "t", "z", "y", "x"])\n        edge_keys = ["edge_id", "source_id", "target_id"]\n        edges = graph.edge_attrs(attr_keys=edge_keys + (["edge_prob", "edge_dist"] if graph.num_edges() else []))\n        return nodes, edges\n\n    @observed\n    def pre_graph(self, coords, edges, node_ids, graph):\n        _require(self.stage == "pre_graph", "pre-graph stage order mismatch")\n        _require(np.array_equal(coords, self.coords) and tuple(tuple(e) for e in edges) == self.edges,\n                 "build_graph inputs changed")\n        _require(len(node_ids) == len(coords) and len(set(node_ids)) == len(coords)\n                 and all(isinstance(i, (int, np.integer)) and not isinstance(i, bool) for i in node_ids),\n                 "nonbijective graph ID mapping")\n        nodes, edge_table = self._graph_tables(graph)\n        node_map = {r["node_id"]: r for r in nodes.iter_rows(named=True)}\n        _require(len(node_map) == len(node_ids), "pre-graph node count mismatch")\n        for row, node_id in enumerate(node_ids):\n            _require(node_id in node_map and\n                     tuple(node_map[node_id][k] for k in ("t", "z", "y", "x")) == tuple(coords[row]),\n                     "graph ID/coordinate mismatch")\n        expected = {}\n        for s, t, prob, dist in edges:\n            _require(type(s) is int and type(t) is int and 0 <= s < len(coords) and 0 <= t < len(coords),\n                     "invalid candidate index")\n            _require(coords[t, 0] == coords[s, 0] + 1 and np.isfinite(prob) and np.isfinite(dist)\n                     and 0 <= prob <= 1 and dist >= 0, "invalid candidate edge")\n            key = (node_ids[s], node_ids[t])\n            _require(key not in expected, "duplicate candidate edge")\n            expected[key] = (prob, dist)\n        actual = {(r["source_id"], r["target_id"]): (r["edge_prob"], r["edge_dist"])\n                  for r in edge_table.iter_rows(named=True)}\n        _require(edge_table.height == len(actual) and actual == expected, "pre-graph candidate mismatch")\n        arrays = {"detector_indices": np.arange(len(coords), dtype=np.int64),\n                  "graph_node_ids": np.asarray(node_ids, dtype=np.int64)}\n        for prefix, table in (("node", nodes), ("edge", edge_table)):\n            arrays.update({f"{prefix}_{column}": table[column].to_numpy() for column in table.columns})\n        self._save_arrays("pre_ilp", arrays)\n        self.before = (node_map, actual)\n        self.stage = "selected_graph"\n\n    @observed\n    def selected_graph(self, graph):\n        _require(self.stage == "selected_graph", "selected-graph stage order mismatch")\n        nodes, edges = self._graph_tables(graph)\n        before_nodes, before_edges = self.before\n        node_ids = nodes["node_id"].to_list()\n        _require(len(node_ids) == len(set(node_ids)), "duplicate selected node")\n        for row in nodes.iter_rows(named=True):\n            _require(before_nodes.get(row["node_id"]) == row, "selected node changed")\n        selected_edges = set()\n        selected_ids = set(node_ids)\n        for row in edges.iter_rows(named=True):\n            key = (row["source_id"], row["target_id"])\n            _require(key not in selected_edges and all(i in selected_ids for i in key)\n                     and before_edges.get(key) == (row["edge_prob"], row["edge_dist"]),\n                     "selected edge changed or dangling")\n            selected_edges.add(key)\n        self._save_arrays("post_ilp", {f"{prefix}_{column}": table[column].to_numpy()\n                                       for prefix, table in (("node", nodes), ("edge", edges))\n                                       for column in table.columns})\n        self.completed.add(self.dataset)\n        self.dataset, self.stage = None, "idle"\n        self.coords = self.edges = self.before = None\n\n    @observed\n    def verify_complete(self):\n        _require(self.dataset is None and self.completed == set(self.pairs.frame_counts), "incomplete graph capture")\n        self.pairs.finish()\n        for item in self.artifacts:\n            path = self.root / item["path"]\n            _require(path.stat().st_size == item["bytes"] and _sha(path) == item["sha256"], "graph artifact changed")\n        self.graph_bytes += (self.pairs.root / "MANIFEST.json").stat().st_size\n\n    def finish(self):\n        self.verify_complete()\n        result = {"status": "ASSOCIATION_OBSERVATION_COMPLETE_NOT_PARITY", "capture_only": True,\n                  "graph_ID_mapping_complete": True, "inference_parity_verified": False,\n                  "submission_authorized": False, "datasets": sorted(self.completed),\n                  "candidate_threshold": .48, "candidate_limits": {"parents": None, "children": None},\n                  "observer_seconds_excluding_final_manifest": self.seconds,\n                  "time_limit_enforcement": "synchronous_call_boundaries",\n                  "pair_manifest_sha256": _sha(self.pairs.root / "MANIFEST.json"),\n                  "graphs": self.artifacts}\n        encoded = _json_bytes(result)\n        try:\n            _require(self.pairs.output_bytes + self.graph_bytes + len(encoded) <= self.pairs.limits.output_bytes,\n                     "observer final manifest byte budget exceeded")\n            with (self.root / "MANIFEST.json").open("xb") as stream:\n                stream.write(encoded)\n        except Exception as exc:\n            self.failed = True\n            with (self.root / "ERROR.json").open("xb") as stream:\n                stream.write(_json_bytes({"status": "ERROR", "error": str(exc), "submission_authorized": False}))\n            raise\n        self.closed = True\n        return json.loads(encoded)\n', 'src/biohub/association_instrumentation.py': '"""Static-only, reversible observer insertion into the pinned E23 reconstruction."""\n\nimport ast\nimport hashlib\n\nSOURCE_SHA256 = "8e7ac19e8b436d6777576b7ea2269464c4ed8842990b45448d40bf178f8f62de"\n\n# Anchors are never replaced. Each addition is removable byte-for-byte.\nINSERTIONS = (\n    ("    edges: list[tuple[int, int, float, float]],\\n", "    association_observer=None,\\n"),\n    ("    secondary_low_margin_max: float = 0.2,\\n", "    association_observer=None,\\n"),\n    ("    evaluate: bool = False,\\n", "    association_observer=None,\\n"),\n    ("    pool_k = pool_kernel_from_um(cfg.pool_kernel_um, voxel_size)\\n",\n     "    if association_observer is not None:\\n"\n     "        association_observer.start_video(\\n"\n     "            ds_path.stem, T, W, downsample, ds.scale, cfg.edge_activation,\\n"\n     "            threshold=cfg.threshold, max_parents=cfg.max_parents_per_node,\\n"\n     "            max_children=cfg.max_children_per_node)\\n"),\n    ("            s_tgt, e_tgt = coord_offset[t_tgt]\\n",\n     "            if association_observer is not None:\\n"\n     "                association_observer.start_pair(t_src, s_src, e_src, s_tgt, e_tgt, coords_so_far)\\n"),\n    ("            )  # (1, n_src, n_tgt)\\n",\n     "            if association_observer is not None:\\n"\n     "                association_observer.primary(edge_logits_pair, unet_feat_src, unet_feat_tgt)\\n"),\n    ("                reverse_logits_pair = reverse_logits_native.transpose(1, 2)\\n",\n     "                if association_observer is not None:\\n"\n     "                    association_observer.reverse(reverse_logits_pair)\\n"),\n    ("                    p_mask_src, p_mask_tgt,\\n                )\\n",\n     "                if association_observer is not None:\\n"\n     "                    association_observer.secondary(\\n"\n     "                        secondary_logits_pair, secondary_feat_src, secondary_feat_tgt)\\n"),\n    ("                probs = torch.sigmoid(raw).cpu().numpy()\\n",\n     "            if association_observer is not None:\\n"\n     "                association_observer.mixed(raw, probs)\\n"),\n    ("    return coords, all_edges\\n",\n     "    if association_observer is not None:\\n"\n     "        association_observer.end_video(coords, all_edges)\\n"),\n    ("    return graph\\n",\n     "    if association_observer is not None:\\n"\n     "        association_observer.pre_graph(coords, edges, node_ids, graph)\\n"),\n    ("                secondary_low_margin_max=secondary_low_margin_max,\\n",\n     "                association_observer=association_observer,\\n"),\n    ("        graph = build_graph(coords, edges)\\n",\n     ""),\n    ("        save_graph(graph, output_dir / f\\"{name}.geff\\")\\n",\n     "        if association_observer is not None:\\n"\n     "            association_observer.selected_graph(graph)\\n"),\n)\n\n\ndef instrument_source(source: str) -> tuple[str, dict]:\n    """Return text and a receipt. Never imports or executes the supplied source.\n\n    The build_graph call needs a keyword, so insert it before the closing paren\n    instead of replacing the call. All original executable AST nodes survive.\n    """\n    digest = hashlib.sha256(source.encode()).hexdigest()\n    if digest != SOURCE_SHA256:\n        raise ValueError("E23 reconstructed source SHA mismatch")\n    patched = source\n    additions = []\n    for index, (anchor, body) in enumerate(INSERTIONS):\n        if patched.count(anchor) != 1:\n            raise ValueError(f"observer anchor {index} is not unique")\n        if not body:  # build_graph keyword insertion, not a replacement function\n            original = "build_graph(coords, edges)"\n            addition = ", association_observer=association_observer"\n            patched = patched.replace(original, original[:-1] + addition + ")", 1)\n            additions.append((original[:-1] + addition + ")", original))\n            continue\n        # End-of-function and post-solver hooks must precede return/save.\n        before = anchor.lstrip().startswith(("return ", "save_graph("))\n        replacement = body + anchor if before else anchor + body\n        patched = patched.replace(anchor, replacement, 1)\n        additions.append((replacement, anchor))\n    restored = patched\n    for inserted, original in reversed(additions):\n        if restored.count(inserted) != 1:\n            raise ValueError("observer removal is ambiguous")\n        restored = restored.replace(inserted, original, 1)\n    if restored != source or ast.dump(ast.parse(restored)) != ast.dump(ast.parse(source)):\n        raise ValueError("original E23 program changed")\n    ast.parse(patched)\n    receipt = {"status": "STATIC_OBSERVER_INSERTION_ONLY", "original_sha256": digest,\n               "instrumented_sha256": hashlib.sha256(patched.encode()).hexdigest(),\n               "restored_bytes_equal": True, "restored_ast_equal": True,\n               "insertions": len(additions), "source_executed": False,\n               "inference_parity_verified": False, "submission_authorized": False}\n    return patched, receipt\n', 'src/biohub/association_parity.py': '"""Common OFF/ON inference bridge and exact observation-parity comparison."""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport importlib.metadata\nimport importlib.util\nimport json\nimport os\nimport random\nimport resource\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom biohub.association_capture import _json_bytes, _require, _sha\nfrom biohub.association_observer import AssociationObserver\n\nE23_ENV = {\n    "BIOHUB_SECONDARY_EDGE_WEIGHT": "0.15", "BIOHUB_SECONDARY_DETECTION_WEIGHT": "0.475",\n    "BIOHUB_SECONDARY_LINK_MODE": "low_margin_consensus", "BIOHUB_SECONDARY_MIX_TEMPERATURE": "1",\n    "BIOHUB_SECONDARY_LOW_MARGIN_MAX": "0.35", "BIOHUB_DUAL_SEED_EDGE_THRESHOLD": "0.48",\n    "BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT": "0.30", "BIOHUB_BIDIRECTIONAL_FUSION_MODE": "harmonic_probability",\n    "BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION": "0.90",\n}\nCFG = {"det_threshold": .96875, "use_ilp": True, "ilp_edge_weight": -1.,\n       "ilp_appearance_weight": 0., "ilp_disappearance_weight": 1.5, "ilp_division_weight": 1.}\n\n\ndef array_signatures(arrays):\n    return {k: {"dtype": a.dtype.str, "shape": list(a.shape),\n                "sha256": hashlib.sha256(a.tobytes(order="C")).hexdigest()} for k, a in sorted(arrays.items())}\n\n\ndef graph_arrays(graph):\n    nodes = graph.node_attrs(attr_keys=["node_id", "t", "z", "y", "x"]).sort("node_id")\n    edge_keys = ["edge_id", "source_id", "target_id"]\n    if graph.num_edges():\n        edge_keys += ["edge_prob", "edge_dist"]\n    edges = graph.edge_attrs(attr_keys=edge_keys).sort("source_id", "target_id", "edge_id")\n    return {f"{prefix}_{column}": table[column].to_numpy()\n            for prefix, table in (("node", nodes), ("edge", edges)) for column in table.columns}\n\n\ndef semantic_graph_signature(arrays):\n    """GEFF has endpoint pairs rather than edge IDs; compare lossless numeric values."""\n    keys = ("node_node_id", "node_t", "node_z", "node_y", "node_x",\n            "edge_source_id", "edge_target_id", "edge_edge_prob", "edge_edge_dist")\n    canonical = {}\n    for k in keys:\n        raw = arrays.get(k, np.empty(0, dtype=np.float64))\n        dtype = np.int64 if k.endswith("_id") or k == "node_t" else np.float64\n        canonical[k] = raw.astype(dtype)\n        _require(np.array_equal(canonical[k], raw), f"lossy graph canonicalization: {k}")\n    return array_signatures(canonical)\n\n\ndef compare_results(off, on):\n    for result, arm in ((off, "off"), (on, "on")):\n        _require(result["status"] == "ARM_COMPLETE" and result["arm"] == arm, "incomplete/wrong arm")\n        names = result["dataset_order"]\n        _require(len(names) == 4 and len(set(names)) == 4 and names == sorted(names), "incomplete dataset coverage")\n        records = result["records"]\n        _require(len(records) == 12 and {(r["dataset"], r["stage"]) for r in records}\n                 == {(name, stage) for name in names for stage in ("returned", "pre", "post")},\n                 "incomplete trace coverage")\n    common = ("dataset_order", "input_inventory", "dependencies", "environment", "model_config", "rng_after")\n    for key in common:\n        _require(off[key] == on[key], f"OFF/ON {key} mismatch")\n    _require(off["plan_sha256"] == on["plan_sha256"], "OFF/ON plan mismatch")\n    _require(off["records"] == on["records"], "OFF/ON prediction/graph arrays mismatch")\n    _require(on["observation_complete"], "ON observer incomplete")\n    return {"status": "PUBLIC4_ASSOCIATION_OBSERVER_PARITY_PASS", "datasets": off["dataset_order"],\n            "reference_raw_matched": True, "submission_authorized": False,\n            "full36_capture_complete": False, "generalization_evidence": False}\n\n\ndef input_inventory(root, datasets):\n    result = []\n    for name in datasets:\n        folder = root / f"{name}.zarr"\n        _require(folder.is_dir(), "missing planned image dataset")\n        for path in sorted(folder.rglob("*")):\n            if path.is_file():\n                result.append({"path": path.relative_to(root).as_posix(),\n                               "bytes": path.stat().st_size, "sha256": _sha(path)})\n    return result\n\n\ndef _rng_digest(torch):\n    numpy_state = np.random.get_state()\n    return {"python": hashlib.sha256(_json_bytes(random.getstate())).hexdigest(),\n            "numpy": hashlib.sha256(numpy_state[1].tobytes() + _json_bytes(\n                [numpy_state[0], *numpy_state[2:]])).hexdigest(),\n            "torch_cpu": hashlib.sha256(torch.get_rng_state().numpy().tobytes()).hexdigest(),\n            "torch_cuda": [hashlib.sha256(s.cpu().numpy().tobytes()).hexdigest()\n                           for s in torch.cuda.get_rng_state_all()]}\n\n\ndef capture_module(module, plan, root, arm):\n    """Run one prepared module. Synthetic tests supply a module-shaped fixture."""\n    _require(arm in ("off", "on"), "unknown arm")\n    names = sorted(plan["datasets"])\n    observer = AssociationObserver(root / "observation", {\n        name: plan["datasets"][name]["frame_counts"] for name in names}) if arm == "on" else None\n    original_video, original_build, original_save = module.predict_video, module.build_graph, module.save_graph\n    records = {}\n    current = None\n    stage = "video"\n    order = []\n\n    def save_record(name, label, arrays):\n        _require((name, label) not in records, "duplicate trace record")\n        _require(all(isinstance(a, np.ndarray) and not a.dtype.hasobject and np.isfinite(a).all()\n                     for a in arrays.values()), "invalid trace array")\n        path = root / f"{name}_{label}.npz"\n        with path.open("xb") as stream:\n            np.savez_compressed(stream, **arrays)\n        with np.load(path, allow_pickle=False) as saved:\n            _require(array_signatures({k: saved[k] for k in saved.files}) == array_signatures(arrays),\n                     "trace roundtrip mismatch")\n        records[name, label] = {"dataset": name, "stage": label, "arrays": array_signatures(arrays)}\n\n    def video(*args, **kwargs):\n        nonlocal current, stage\n        _require(stage == "video", "overlapping prediction video")\n        name = Path(args[1]).stem\n        _require(name in names and name not in order, "unplanned or duplicate prediction video")\n        current = name\n        coords, edges = original_video(*args, **kwargs)\n        expected = plan["datasets"][name]\n        _require(coords.dtype == np.dtype("int16") and coords.shape == (sum(expected["frame_counts"]), 4),\n                 "reference detection shape/dtype mismatch")\n        _require(hashlib.sha256(coords.astype("<i2", copy=False).tobytes()).hexdigest()\n                 == expected["coordinate_sha256"], "reference detection coordinate mismatch")\n        _require([int(np.count_nonzero(coords[:, 0] == t)) for t in range(100)] == expected["frame_counts"],\n                 "reference detector counts mismatch")\n        save_record(name, "returned", {"coords": coords, "edges": np.array(edges, dtype=np.float64).reshape(-1, 4)})\n        order.append(name)\n        stage = "pre"\n        return coords, edges\n\n    def build(*args, **kwargs):\n        nonlocal stage\n        _require(stage == "pre", "unexpected graph build")\n        graph = original_build(*args, **kwargs)\n        save_record(current, "pre", graph_arrays(graph))\n        stage = "post"\n        return graph\n\n    def save(graph, path, *args, **kwargs):\n        nonlocal stage\n        _require(stage == "post" and Path(path).stem == current, "unexpected graph save")\n        arrays = graph_arrays(graph)\n        _require(semantic_graph_signature(arrays) == plan["datasets"][current]["raw_graph_signature"],\n                 "reference selected raw graph mismatch")\n        save_record(current, "post", arrays)\n        result = original_save(graph, path, *args, **kwargs)\n        stage = "video"\n        return result\n\n    module.predict_video, module.build_graph, module.save_graph = video, build, save\n    try:\n        kwargs = {"data_dir": Path(plan["data_dir"]), "fold": 0, "splits_file": Path(plan["splits_file"]),\n                  "weights_path": Path(plan["primary_weights"]), "cfg": module.PredictConfig(**CFG),\n                  "method": "unet_transformer", "unet_batch_size": 4, "evaluate": False}\n        if observer is not None:\n            kwargs["association_observer"] = observer\n        module.predict(**kwargs)\n        _require(stage == "video" and order == names and len(records) == len(names) * 3, "incomplete trace")\n        if observer is not None:\n            observer.finish()\n        return {"dataset_order": order, "records": [records[key] for key in sorted(records)],\n                "observation_complete": observer is not None}\n    finally:\n        module.predict_video, module.build_graph, module.save_graph = original_video, original_build, original_save\n\n\ndef run_arm(plan_path, source, root, arm):\n    """Kaggle-only entry. Fail closed before importing a source with unverified bytes."""\n    _require(Path("/kaggle/working").is_dir(), "real model execution is Kaggle-only")\n    plan = json.loads(plan_path.read_text())\n    _require(plan["schema_version"] == "biohub.association.public4.v1" and len(plan["datasets"]) == 4,\n             "invalid public4 plan")\n    _require(arm in ("off", "on") and _sha(source) == plan["source_sha256"][arm], "source SHA mismatch")\n    _require({key: os.environ.get(key) for key in E23_ENV} == E23_ENV, "effective E23 environment mismatch")\n    _require(set(p.stem for p in Path(plan["data_dir"]).glob("*.zarr")) == set(plan["datasets"]),\n             "public dataset set mismatch")\n    for path, expected in plan["runtime_file_sha256"].items():\n        _require(_sha(Path(path)) == expected, f"runtime file changed: {path}")\n    for record in plan["source_inventory"]:\n        _require(_sha(Path(record["path"])) == record["sha256"], "source inventory changed")\n    root.mkdir(parents=True, exist_ok=False)\n    started = time.perf_counter()\n    try:\n        inputs = input_inventory(Path(plan["data_dir"]), sorted(plan["datasets"]))\n        import torch\n        _require(torch.cuda.is_available() and torch.cuda.device_count() == 1, "expected one visible CUDA device")\n        # Common diagnostic seed; E23 saved raw parity must still pass, no baseline replacement.\n        random.seed(23826)\n        np.random.seed(23826)\n        torch.manual_seed(23826)\n        torch.cuda.manual_seed_all(23826)\n        dependencies = {name: importlib.metadata.version(name) for name in\n                        ("numpy", "torch", "polars", "tracksdata", "pyscipopt", "zarr", "geff", "scipy")}\n        spec = importlib.util.spec_from_file_location("biohub_e23_target_predict", source)\n        module = importlib.util.module_from_spec(spec)\n        sys.modules[spec.name] = module\n        spec.loader.exec_module(module)\n        import dataspec\n        # The original predictor deletes an existing output; point it only at this fresh arm.\n        dataspec.PREDICTIONS_PATH = root / "raw_predictions"\n        torch.cuda.synchronize()\n        result = capture_module(module, plan, root, arm)\n        torch.cuda.synchronize()\n        _require(inputs == input_inventory(Path(plan["data_dir"]), sorted(plan["datasets"])), "image bytes changed")\n        for path, expected in plan["runtime_file_sha256"].items():\n            _require(_sha(Path(path)) == expected, "runtime file changed after inference")\n        for record in plan["source_inventory"]:\n            _require(_sha(Path(record["path"])) == record["sha256"], "source changed after inference")\n        _require(_sha(source) == plan["source_sha256"][arm], "prediction source changed")\n        result.update(status="ARM_COMPLETE", arm=arm, input_inventory=inputs, dependencies=dependencies,\n                      environment=E23_ENV, model_config=CFG, rng_after=_rng_digest(torch),\n                      plan_sha256=_sha(plan_path), source_sha256=_sha(source), submission_authorized=False,\n                      wall_seconds=time.perf_counter() - started,\n                      peak_cuda_allocated_bytes=torch.cuda.max_memory_allocated(),\n                      peak_cuda_reserved_bytes=torch.cuda.max_memory_reserved(),\n                      peak_self_rss_bytes=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * 1024)\n        _require(result["wall_seconds"] <= 3600 and result["peak_self_rss_bytes"] <= 24 * 1024**3,\n                 "arm runtime/RSS budget exceeded")\n        _require(sum(p.stat().st_size for p in root.rglob("*") if p.is_file()) <= 12 * 1024**3,\n                 "arm output budget exceeded")\n        with (root / "RESULT.json").open("xb") as stream:\n            stream.write(_json_bytes(result))\n        print(json.dumps({"status": result["status"], "arm": arm, "wall_seconds": result["wall_seconds"]}), flush=True)\n        return result\n    except Exception as exc:\n        with (root / "ERROR.json").open("xb") as stream:\n            stream.write(_json_bytes({"status": "ERROR", "error": str(exc), "submission_authorized": False}))\n        raise\n', 'src/biohub/association_supervise.py': '"""Serial bounded Kaggle OFF/ON supervision; never submits or retries."""\n\nimport json\nimport os\nimport shutil\nimport signal\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\nfrom biohub.association_capture import _json_bytes, _require, _sha\nfrom biohub.association_parity import compare_results\n\n\ndef process_tree_rss(pid, proc_root=Path("/proc")):\n    """Linux resident-page sampling, following children of every thread, no extra package."""\n    pending, seen, pages = [pid], set(), 0\n    while pending:\n        current = pending.pop()\n        if current in seen:\n            continue\n        seen.add(current)\n        folder = proc_root / str(current)\n        try:\n            pages += int((folder / "statm").read_text().split()[1])\n            for children in (folder / "task").glob("*/children"):\n                pending.extend(int(value) for value in children.read_text().split())\n        except FileNotFoundError:\n            continue  # Process/thread ended between samples; exit status is checked separately.\n    return pages * os.sysconf("SC_PAGE_SIZE")\n\n\ndef supervise(plan_path, root, payload, deadline):\n    _require(Path("/kaggle/working").is_dir(), "supervisor is Kaggle-only")\n    root.mkdir(parents=True, exist_ok=False)\n    _require(shutil.disk_usage(root).free >= 16 * 1024**3, "less than 16 GiB free")\n    plan = json.loads(plan_path.read_text())\n    completed = {}\n    process_records = {}\n    tokens = [t.strip() for t in os.environ.get("CUDA_VISIBLE_DEVICES", "0").split(",") if t.strip()]\n    _require(tokens and tokens[0] != "-1", "no CUDA token")\n    plan_sha = _sha(plan_path)\n    for arm in ("off", "on"):\n        _require(time.monotonic() < deadline, "diagnostic wall budget exceeded before arm")\n        command = [sys.executable, str(payload / "scripts/e23_association_bridge.py"),\n                   "--plan", str(plan_path), "--source", plan["source_paths"][arm],\n                   "--root", str(root / arm), "--arm", arm]\n        env = {**os.environ, "PYTHONPATH": str(payload / "src") + os.pathsep + str(Path(plan["repo_dir"]) / "src"),\n               "CUDA_VISIBLE_DEVICES": tokens[0], "BIOHUB_GPU_SHARD": "single",\n               "PYTHONHASHSEED": "0"}\n        started, peak_rss = time.monotonic(), 0\n        error = None\n        with (root / f"{arm}.log").open("xb") as log:\n            process = subprocess.Popen(command, cwd=plan["repo_dir"], env=env, stdout=log,\n                                       stderr=subprocess.STDOUT, start_new_session=True)\n            print(f"D3 ARM_STARTED {arm} PID {process.pid}", flush=True)\n            try:\n                while process.poll() is None:\n                    if time.monotonic() >= min(started + 3600, deadline):\n                        raise RuntimeError("arm/diagnostic wall budget exceeded")\n                    peak_rss = max(peak_rss, process_tree_rss(process.pid))\n                    _require(peak_rss <= 24 * 1024**3, "process-tree RSS budget exceeded")\n                    size = sum(p.stat().st_size for p in (root / arm).rglob("*") if p.is_file())\n                    _require(size <= 12 * 1024**3, "arm output budget exceeded")\n                    time.sleep(1.)\n            except Exception as exc:\n                error = str(exc)\n                if process.poll() is None:\n                    os.killpg(process.pid, signal.SIGTERM)\n                    try:\n                        process.wait(timeout=15)\n                    except subprocess.TimeoutExpired:\n                        os.killpg(process.pid, signal.SIGKILL)\n                        process.wait()\n            code = process.wait()\n        process_records[arm] = {"returncode": code, "wall_seconds": time.monotonic() - started,\n                                "peak_sampled_tree_rss_bytes": peak_rss, "error": error}\n        with (root / f"{arm}_PROCESS.json").open("xb") as stream:\n            stream.write(_json_bytes(process_records[arm]))\n        print(f"D3 ARM_FINISHED {arm} exit={code}", flush=True)\n        _require(code == 0 and error is None, "arm failed; inspect retained log/PROCESS/ERROR")\n        _require(_sha(plan_path) == plan_sha, "plan changed during diagnostic")\n        completed[arm] = json.loads((root / arm / "RESULT.json").read_text())\n    result = compare_results(completed["off"], completed["on"])\n    result.update(processes=process_records, plan_sha256=plan_sha,\n                  arm_result_sha256={arm: _sha(root / arm / "RESULT.json") for arm in completed})\n    _require(time.monotonic() <= deadline, "diagnostic deadline exceeded")\n    with (root / "PARITY_RESULT.json").open("xb") as stream:\n        stream.write(_json_bytes(result))\n    print(json.dumps(result, sort_keys=True), flush=True)\n    return result\n', 'scripts/e23_association_bridge.py': '"""Run one frozen public-four E23 observation arm, only inside Kaggle."""\n\nimport argparse\nfrom pathlib import Path\n\nfrom biohub.association_parity import run_arm\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--plan", type=Path, required=True)\n    parser.add_argument("--source", type=Path, required=True)\n    parser.add_argument("--root", type=Path, required=True)\n    parser.add_argument("--arm", choices=("off", "on"), required=True)\n    args = parser.parse_args()\n    run_arm(args.plan, args.source, args.root, args.arm)\n\n\nif __name__ == "__main__":\n    main()\n', 'src/biohub/__init__.py': '# Diagnostic-only payload namespace.\n'}
for relative, text in D3_FILES.items():
    target = D3_PAYLOAD / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    with target.open("x") as stream:
        stream.write(text)
sys.path.insert(0, str(D3_PAYLOAD / "src"))
from biohub.association_instrumentation import instrument_source
from biohub.association_capture import _sha
patched, receipt = instrument_source(_ps.read_text())
assert receipt["instrumented_sha256"] == 'e2fda47e41d970a2c650bbbbb5fe068b6e1b9667fdd91cb59cc2a3ac8ba95a9c'
D3_SOURCES = {arm: REPO_DIR / "scripts" / f"e23_observer_{arm}.py" for arm in ("off", "on")}
for arm, text in (("off", _ps.read_text()), ("on", patched)):
    with D3_SOURCES[arm].open("x") as stream:
        stream.write(text)
D3_SPLITS = WORKING_DIR / "association_public4_splits.json"
with D3_SPLITS.open("x") as stream:
    json.dump([{"split": 0, "train": [], "test": D3_EXPECTED_DATASETS}], stream)
runtime_files = {str(_primary_materialized_path): _primary_expected_sha256,
                 str(SECONDARY_WEIGHTS_PATH): _secondary_expected_sha256,
                 str(_deepcenter_materialized_path): _deepcenter_expected_sha256,
                 str(_primary_materialized_path.parent / "config.json"): 'e9b4e396c58081bca08adf8275bd0bd1c2d3fd6eb091a1912a5116cb6de7b50a',
                 str(SECONDARY_CONFIG_PATH): 'e9b4e396c58081bca08adf8275bd0bd1c2d3fd6eb091a1912a5116cb6de7b50a',
                 str(D3_SPLITS): _sha(D3_SPLITS)}
for path, expected in runtime_files.items():
    assert _sha(Path(path)) == expected, path
D3_PLAN = {"schema_version": "biohub.association.public4.v1", "datasets": {'44b6_0113de3b': {'coordinate_sha256': '257fd2188e49115c84dbe00a5439abfcc0b79809cc2ccfcc0b994099eee8188d', 'frame_counts': [221, 224, 228, 222, 219, 221, 223, 229, 225, 222, 226, 225, 230, 234, 236, 235, 222, 226, 225, 230, 233, 237, 234, 237, 240, 239, 237, 240, 240, 250, 251, 254, 257, 258, 262, 265, 261, 260, 255, 256, 254, 255, 259, 267, 270, 266, 272, 272, 273, 271, 269, 294, 295, 296, 290, 285, 288, 282, 285, 290, 291, 292, 297, 285, 295, 290, 292, 286, 287, 285, 289, 288, 285, 286, 284, 280, 285, 286, 283, 288, 293, 290, 286, 281, 285, 282, 284, 284, 285, 281, 289, 285, 278, 279, 272, 269, 270, 266, 263, 258], 'raw_graph_signature': {'edge_edge_dist': {'dtype': '<f8', 'shape': [24901], 'sha256': '5209c987a6c5b59f7c9601ff3e21961a572cb87c7bca29e9166538d6f2b9baec'}, 'edge_edge_prob': {'dtype': '<f8', 'shape': [24901], 'sha256': '573db8001b757bf8f0b08c50673b2bb4b77ea19de7b110ca512ec7e7a73f7789'}, 'edge_source_id': {'dtype': '<i8', 'shape': [24901], 'sha256': 'd3ea3a88f9d12239182a51cebbb7408f61a11ca01f1966565c84aa0663688fa8'}, 'edge_target_id': {'dtype': '<i8', 'shape': [24901], 'sha256': 'e0f31aad6eed341a70750998b344d7bdc38fa6246efd8ac7f5d733546962f439'}, 'node_node_id': {'dtype': '<i8', 'shape': [25828], 'sha256': 'bbe142826731d6ca5a668a62b2c65a4ad23be0952bd60c193ac6ef86252a7f83'}, 'node_t': {'dtype': '<i8', 'shape': [25828], 'sha256': 'aa678d51378e8d0a7556fda92818925cf44c51c32ae2a6c019e5698a78b60e28'}, 'node_x': {'dtype': '<f8', 'shape': [25828], 'sha256': 'dbcb108f2d680299f204d436afe8a588ad09d10bf5f23bcc417ff4e599957fac'}, 'node_y': {'dtype': '<f8', 'shape': [25828], 'sha256': '9c2d7fcfd18ac6cbea9bc3e777c529e907a0ec70ad398ae11cb7e4113ad5585c'}, 'node_z': {'dtype': '<f8', 'shape': [25828], 'sha256': 'fe9a4ce15c31d2b39833cacefc993dda7db50330f15a3379bff5d20b0641e7fc'}}}, '6bba_05b6850b': {'coordinate_sha256': 'fdad43c29c1a43397982fed5c4c5c7e144517d26bf5a487680efa3b897a2bb19', 'frame_counts': [78, 81, 81, 79, 77, 75, 72, 75, 71, 82, 76, 73, 78, 76, 72, 73, 75, 80, 75, 76, 68, 62, 65, 65, 68, 68, 64, 61, 63, 65, 59, 59, 57, 62, 61, 57, 56, 54, 66, 64, 63, 66, 67, 69, 73, 70, 70, 71, 73, 74, 77, 79, 80, 83, 90, 90, 88, 80, 81, 77, 76, 77, 74, 71, 82, 73, 74, 73, 73, 73, 74, 83, 69, 73, 72, 71, 73, 71, 71, 68, 71, 78, 79, 72, 75, 71, 73, 71, 73, 69, 68, 71, 61, 65, 70, 66, 65, 68, 66, 68], 'raw_graph_signature': {'edge_edge_dist': {'dtype': '<f8', 'shape': [6035], 'sha256': 'ecace66e2b86ba81685260aa46446c297c0ef3c82e981c9480ecb83e27a16abd'}, 'edge_edge_prob': {'dtype': '<f8', 'shape': [6035], 'sha256': '80af7dafa4a811d285508992644bc6e5e26d8082131ed15bdbdeb8ec548b5e71'}, 'edge_source_id': {'dtype': '<i8', 'shape': [6035], 'sha256': 'a1b0bfb3568a809818e0c295bfd8d3677e913c7cbd9ceda90db154071c28aeac'}, 'edge_target_id': {'dtype': '<i8', 'shape': [6035], 'sha256': 'd2376bc86716b26b28d95c8a3a5fddaa87be0018e6bf71665933b6fd6dff57dd'}, 'node_node_id': {'dtype': '<i8', 'shape': [6361], 'sha256': 'a745482919fb715efbebcc6a8dc6c646ccca39bb210ac7f4854e49268fa8286c'}, 'node_t': {'dtype': '<i8', 'shape': [6361], 'sha256': 'c79912a2e2474816db162354aae9dc9cee57162fe535b116ae588ed8ccddffa6'}, 'node_x': {'dtype': '<f8', 'shape': [6361], 'sha256': 'd05fbb92082d2afedd32250e7d5ad10cdac802f24f088f980f9b1a7b50d8e032'}, 'node_y': {'dtype': '<f8', 'shape': [6361], 'sha256': 'bef059e84d15ed1512d29c1c381d1c3ddc946ae0de0321127fad2746cef6167e'}, 'node_z': {'dtype': '<f8', 'shape': [6361], 'sha256': 'd0f9c06ba280f52c10f725a8bc63e76b244861e4e2bc2aa774963e1fce84dad7'}}}, '44b6_0b24845f': {'coordinate_sha256': '07a5918d634d554d184516047d1b38196c466bbc4233b87cf284f88fe73bf555', 'frame_counts': [456, 437, 387, 419, 395, 378, 390, 380, 410, 384, 412, 396, 354, 380, 336, 343, 357, 336, 352, 319, 313, 316, 317, 320, 301, 294, 283, 299, 296, 316, 302, 269, 289, 242, 288, 326, 345, 302, 319, 297, 311, 307, 269, 253, 276, 310, 270, 304, 310, 301, 253, 356, 339, 305, 327, 353, 431, 296, 308, 342, 384, 320, 340, 309, 332, 333, 369, 306, 320, 318, 298, 318, 249, 257, 285, 367, 369, 323, 388, 420, 382, 378, 377, 368, 359, 289, 276, 314, 356, 312, 266, 232, 283, 249, 283, 295, 268, 316, 284, 214], 'raw_graph_signature': {'edge_edge_dist': {'dtype': '<f8', 'shape': [19625], 'sha256': '505598249ee53784a8e460f256041d34b71215e6eb770ec8828e77e0aa0af554'}, 'edge_edge_prob': {'dtype': '<f8', 'shape': [19625], 'sha256': 'b1bcb02dde774984955fef9c4b3e4932c6a52f61e0f67ca07d31c9be9eb1debf'}, 'edge_source_id': {'dtype': '<i8', 'shape': [19625], 'sha256': 'd77df87a8735190f69dcf84453330365ee26f4db50efa66cbd9cd389d2fcd621'}, 'edge_target_id': {'dtype': '<i8', 'shape': [19625], 'sha256': '937d1918174ba84ff5ed241bb76b8117630b3828f2be5a86f8bf017d5397bea3'}, 'node_node_id': {'dtype': '<i8', 'shape': [22447], 'sha256': 'a9b4b89c20a9f02c3fab3d699b241be1ed5a4e2cef311cd794dbaf67a4b6dd70'}, 'node_t': {'dtype': '<i8', 'shape': [22447], 'sha256': '362e043e3ca759087d21ec6dca6738a706bc989bc9d3d98545d21d12d637ffb1'}, 'node_x': {'dtype': '<f8', 'shape': [22447], 'sha256': '770a804aeef79686a34d6e9530ec172324a0d91d35767403426627892902acce'}, 'node_y': {'dtype': '<f8', 'shape': [22447], 'sha256': '478bc061c3d47a486aba2bbd57f556d3b582e00282983888834bb8cab7656488'}, 'node_z': {'dtype': '<f8', 'shape': [22447], 'sha256': '3049bd76f945b2d186727d8163b72096e4fa84099ac2c5ce4e583f06d41f5009'}}}, '6bba_05db0fb1': {'coordinate_sha256': 'a6baaf9d4b51ec6109e35f84fb5e1106e0ae01648b1f29da82afc2c9421a2d7a', 'frame_counts': [791, 793, 791, 781, 792, 793, 796, 807, 797, 791, 792, 822, 792, 799, 811, 805, 826, 813, 823, 819, 836, 825, 826, 827, 814, 824, 838, 812, 845, 811, 815, 835, 822, 825, 815, 809, 826, 819, 809, 793, 825, 796, 791, 782, 788, 786, 765, 763, 773, 777, 761, 756, 747, 738, 740, 739, 744, 742, 730, 743, 732, 723, 732, 723, 733, 728, 733, 715, 709, 710, 711, 707, 702, 709, 688, 682, 705, 683, 681, 694, 671, 711, 673, 695, 679, 677, 693, 694, 713, 676, 680, 684, 676, 710, 727, 691, 679, 671, 662, 648], 'raw_graph_signature': {'edge_edge_dist': {'dtype': '<f8', 'shape': [67648], 'sha256': '4ffbb62c89a4572d5d1c8618844bb9e2b898ebeb7d02e8a0aab98d3d9ad75c0a'}, 'edge_edge_prob': {'dtype': '<f8', 'shape': [67648], 'sha256': '14ac455af97b625f987941786c67a85d6418ca6fcffc2060e02fcb19836c0845'}, 'edge_source_id': {'dtype': '<i8', 'shape': [67648], 'sha256': '32b1ee0bf2ad664351e60229ec76296b8ff5b709c47132db3ae6a9710713928c'}, 'edge_target_id': {'dtype': '<i8', 'shape': [67648], 'sha256': '20e7ce3906c84baa20407dbf6b5a9410600a34877663579137467687d46b4208'}, 'node_node_id': {'dtype': '<i8', 'shape': [71491], 'sha256': '1d2320f84c17f9e70d2ac1378e2bdc837bbdcc867e7521faf9c5d25d670e0bf4'}, 'node_t': {'dtype': '<i8', 'shape': [71491], 'sha256': 'fe0932f6099a422da703159437a6680619b3b91df5a73b6ed761b46b4786deb2'}, 'node_x': {'dtype': '<f8', 'shape': [71491], 'sha256': '3f3c150ead4746e600d94eec6e0c53f90c7ee180048262fc31d90f0001c5230e'}, 'node_y': {'dtype': '<f8', 'shape': [71491], 'sha256': 'e43b809fccd7574b16264b1375796a3de11c22fd99a4e1480d3c01a7a569b83c'}, 'node_z': {'dtype': '<f8', 'shape': [71491], 'sha256': 'c2070a94ae43f1e404d821e901215e5d4867e2b29c81767e5027f5bf408a7746'}}}},
           "data_dir": str(TEST_DIR), "splits_file": str(D3_SPLITS), "repo_dir": str(REPO_DIR),
           "primary_weights": str(_primary_materialized_path),
           "runtime_file_sha256": runtime_files,
           "source_paths": {arm: str(path) for arm, path in D3_SOURCES.items()},
           "source_sha256": {"off": receipt["original_sha256"], "on": receipt["instrumented_sha256"]},
           "source_inventory": [{"path": str(p), "sha256": _sha(p)}
                                for p in sorted([*REPO_DIR.rglob("*.py"), *D3_PAYLOAD.rglob("*.py")])],
           "submission_authorized": False}
D3_PLAN_PATH = WORKING_DIR / "association_public4_plan.json"
with D3_PLAN_PATH.open("x") as stream:
    json.dump(D3_PLAN, stream, sort_keys=True, indent=2)
print("D3 PLAN SEALED", _sha(D3_PLAN_PATH), flush=True)


In [ ]:
from biohub.association_supervise import supervise
D3_RESULT = supervise(D3_PLAN_PATH, WORKING_DIR / "association_parity_run", D3_PAYLOAD, D3_STARTED + 8400)
assert D3_RESULT["status"] == "PUBLIC4_ASSOCIATION_OBSERVER_PARITY_PASS"
assert not (WORKING_DIR / "submission.csv").exists()
